In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, BoundaryNorm
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from osgeo import gdal
import rasterio

import pickle

from math import radians, cos, sin, sqrt, atan2

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
from sklearn.metrics import d2_absolute_error_score
from sklearn.metrics import mean_squared_error
# from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import explained_variance_score
from sklearn.metrics import max_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.cluster import HDBSCAN
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from matplotlib.colors import Normalize
from matplotlib.colors import ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
from shapely.geometry import Point

import random

import libpysal as lp
from libpysal.weights import DistanceBand
from esda.moran import Moran

# from mpl_toolkits.mplot3d import Axes3D
# from mpl_toolkits.basemap import Basemap

# For sanity checking
from sklearn.decomposition import PCA

from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay

from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.stats import gaussian_kde
from scipy.spatial.distance import cdist
from scipy.ndimage import label
from scipy.spatial import cKDTree

import glob

from geopy.distance import geodesic

from joblib import dump, load
from tqdm.notebook import tqdm

import os
import shutil
# import pysplit
import time

import warnings
from geopy.distance import geodesic

# import umap
from geopy.distance import great_circle
from joblib import Parallel, delayed
# from haversine import haversine, Unit
# from numba import jit

# plt.ioff()

In [ ]:
# Define a function that calcualte error metrics from predicted and actual values
def reg_model_metrics(actual,pred):
    MSE = mean_squared_error(actual,pred)
    RMSE = np.sqrt(MSE)
    actual_mean = np.mean(actual)
    RRMSE = 100*RMSE/actual_mean
    MAE = mean_absolute_error(actual, pred)
    R2 = r2_score(actual,pred)
    D2 = d2_absolute_error_score(actual, pred)
    MAXErr = max_error(actual, pred)
    EVS = explained_variance_score(actual, pred)
    return MSE, RMSE, RRMSE, MAE, R2, D2, MAXErr, EVS 

# Function to calculate and organize metrics
def calculate_and_organize_metrics_cluster(actual, pred, cluster, label, train_test=""):
     
    # Calculate standard error metrics
    mse, rmse, rrmse, mae, r2, d2, max_err, evs = reg_model_metrics(actual, pred)
    
    # Organize metrics into a dictionary
    metrics_dict = {
        "ATS": label,
        "Cluster": cluster,
        f"MSE_{train_test}": mse,
        f"RMSE_{train_test}": rmse,
        f"RRMSE_{train_test}": rrmse,
        f"MAE_{train_test}": mae,
        f"R2_{train_test}": r2,
        f"D2_{train_test}": d2,
        f"MaxErr_{train_test}": max_err,
        f"EVS_{train_test}": evs,
    }
    
    # Convert dictionary to DataFrame
    metrics_df = pd.DataFrame(metrics_dict, index=[0])
   
    return metrics_df

# Function to calculate and organize metrics
def calculate_and_organize_metrics_global(actual, pred, label):
     
    # Calculate standard error metrics
    mse, rmse, rrmse, mae, r2, d2, max_err, evs = reg_model_metrics(actual, pred)
    
    # Organize metrics into a dictionary
    metrics_dict = {
        "ATS": label,
        f"MSE": mse,
        f"RMSE": rmse,
        f"RRMSE": rrmse,
        f"MAE": mae,
        f"R2": r2,
        f"D2": d2,
        f"MaxErr": max_err,
        f"EVS": evs,
    }
    
    # Convert dictionary to DataFrame
    metrics_df = pd.DataFrame(metrics_dict, index=[0])
   
    return metrics_df


def scatter_plot(actual, pred, title, var_of_int, unit_label, comp_flag, save_path):
    # Color Matching for clarity
    if var_of_int == 'CH4':
        color = 'blue'
        cm = 'Blues'
    elif var_of_int == 'DMS':
        color = 'purple'
        cm = 'Purples'
    elif var_of_int == 'CO':
        color = 'cyan'
        cm = 'YlGnBu'
    elif var_of_int == 'O3':
        color = 'red'
        cm = 'YlOrRd'
    elif var_of_int == 'CH3Br':
        color = 'mediumseagreen'
        cm = 'Greens'
    elif var_of_int == 'Ethane':
        color = 'orange'
        cm = 'Oranges'
    else:
        color = 'gray'
        cm = 'Greys'
        
    if comp_flag:
        cm = "Greys"
    
    MSE, RMSE, RRMSE, MAE, R2, D2, MAXErr, EVS = reg_model_metrics(actual, pred)
        
    fig,ax = plt.subplots(figsize=(8, 6))
    ax.scatter(actual, pred, edgecolors=(0,0,0), c=color, cmap=cm)
    ax.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--', lw=2)
    text = r"R2 = %.2f" % (R2); text += "\n";
    text += r"D2 = %.2f" % (D2); text += "\n";
    text += r"MAE = %.2f" % (MAE); text += "\n";
    text += r"MSE = %.2f" % (MSE); text += "\n";
    text += r"RMSE = %.2f" % (RMSE);     
    plt.annotate(text, xy=(0.01, 0.9), xycoords='axes fraction',color='black', fontsize=10,bbox=dict(facecolor='none', edgecolor='none'))
    ax.set_xlabel(f'Measured {var_of_int} {unit_label}')
    ax.set_ylabel(f'Predicted {var_of_int} {unit_label}')
    # Set log scale for x and y axes
    ax.set_xscale('log')
    ax.set_yscale('log')
    # if var_of_int != 'DMS':
    # Create locators to specify the ticks on both axes
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))  # Controls major ticks
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs="auto"))  # Controls minor ticks

    ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
    ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs="auto"))

    # Ensure minor gridlines are visible and major gridlines are configured
    ax.grid(visible=True, which="major", linestyle="--", linewidth=0.5)  # Major gridlines
    ax.grid(visible=True, which="minor", linestyle=":", linewidth=0.5)  # Minor gridlines
    ax.set_title(title)
    # plt.grid()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    # plt.show()
    plt.clf()
    plt.close()
    return None

    
    # fig,ax = plt.subplots(figsize=(8, 6))
    # ax.scatter(actual, pred, edgecolors=(0,0,0), c=color, cmap=cm)
    # ax.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--', lw=2)
    # text = r"W_R2 = %.2f" % (WR2); text += "\n"; text += r"W_MAE = %.2f" % (WMAE); text += "\n"; text += r"W_MSE = %.2f" % (WMSE); text += "\n"; text += r"W_RMSE = %.2f" % (WRMSE);      
    # plt.annotate(text, xy=(0.05, 0.85), xycoords='axes fraction',color='black', fontsize=10,
    #              bbox=dict(facecolor='none', edgecolor='none'))
    # ax.set_xlabel('Measured ' + str.split(var_of_int)[0] + time + " " + label)
    # ax.set_ylabel('Predicted ' + str.split(var_of_int)[0] + time + " " + label)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_title(title)
    # plt.savefig("C:/Users/vwgei/Documents/PASTEL/plots/" + m + "/Scatter_" + m + sl + str.split(var_of_int)[0] + time + stri + ".png")
    # plt.show()
    
# Permutation importance calclation and plot
def plot_permutation_importance_scores(input_model, input_x, input_y, title, save_path, rs=42):
    xl = input_x.columns # input_x labels
    # yl = str.split(input_y.columns[0])[0] # #this line for not smogn
    yl = input_y.name # This line for smogn
    
    # Calculate the Variable Importance
    perm_imp = permutation_importance(input_model, input_x, input_y, n_repeats=10, random_state=rs, n_jobs=-1)
    # Sort inices
    sorted_idx = perm_imp.importances_mean.argsort()

    # Plot a figure
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.boxplot(perm_imp.importances[sorted_idx].T,
               vert=False,
               labels=xl[sorted_idx])
    ax.set_title(title)
    fig.tight_layout()
    plt.savefig(save_path,  dpi=300, bbox_inches='tight')
    # plt.show()
    plt.clf()
    plt.close()
    return perm_imp, sorted_idx

# Working dot and lines plot
def plot_average_permutation_importance_scores_1(models, input_x, input_y, title, save_path, rs=42):
    # List to store permutation importance results for each model
    perm_importances = []

    # Calculate permutation importance for each model
    for model in models:
        perm_imp = permutation_importance(model, input_x, input_y, n_repeats=10, random_state=rs, n_jobs=-1)
        perm_importances.append(perm_imp.importances)

    # Calculate mean and std for each feature across all models
    perm_importances = np.array(perm_importances)  # Shape (n_models, n_features, n_repeats)
    mean_importances = perm_importances.mean(axis=(0, 2))  # Average across models and repeats
    std_importances = perm_importances.std(axis=(0, 2))    # Std across models and repeats
    sorted_idx = np.argsort(mean_importances)  # Sort indices for plotting

    # Plot permutation importances
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.errorbar(mean_importances[sorted_idx], range(len(mean_importances)), 
                xerr=std_importances[sorted_idx], fmt='o')
    ax.set_yticks(range(len(mean_importances)))
    ax.set_yticklabels(input_x.columns[sorted_idx])
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path,  dpi=300, bbox_inches='tight')
    plt.clf()
    plt.close()

    # Return calculated values
    return mean_importances, std_importances, sorted_idx

# Working boxplot for average permutation importance scores
def plot_average_permutation_importance_scores_2(models, input_x, input_y, title, save_path, rs=42):
    # List to store permutation importance results for each model
    perm_importances = []

    # Calculate permutation importance for each model
    for model in models:
        perm_imp = permutation_importance(model, input_x, input_y, n_repeats=10, random_state=rs, n_jobs=-1)
        perm_importances.append(perm_imp.importances)

    # Convert list to a numpy array of shape (n_models, n_features, n_repeats)
    perm_importances = np.array(perm_importances)

    # Calculate mean importance for sorting
    mean_importances = perm_importances.mean(axis=(0, 2))  # Mean across models and repeats
    sorted_idx = np.argsort(mean_importances)  # Sort indices based on mean importance

    # Reorder the importances and keep the shape consistent with labels
    perm_importances_sorted = perm_importances[:, sorted_idx, :]

    # Plot permutation importances as a boxplot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Reshape to (n_features, n_models * n_repeats) for boxplot compatibility
    perm_importances_flat = perm_importances_sorted.reshape(-1, len(models) * perm_importances.shape[2])

    # Boxplot for each feature's permutation importances
    ax.boxplot(perm_importances_flat.T, vert=False, labels=input_x.columns[sorted_idx])

    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.clf()
    plt.close()

    return perm_importances, sorted_idx


# Working 1D
# def plot_ensemble_partial_dependence(models, X, top_features, save_path, title="Ensemble Partial Dependence"):
#     """
#     Generate partial dependence plots for an ensemble of models on the top two features.
    
#     Args:
#         models (list): List of trained models.
#         X (pd.DataFrame): DataFrame with features for plotting PDP.
#         top_features (list): List of top two features for PDP.
#         save_path (str): Path to save the plot.
#         title (str): Title for the plot.
#     """
    
#     try:
#         fig, ax = plt.subplots(ncols=3, figsize=(10, 4), constrained_layout=True)
        
#         # For each model in the ensemble, compute and plot the partial dependence
#         for model in models:
#             PartialDependenceDisplay.from_estimator(
#                 model,
#                 X,
#                 features=[top_features[0], top_features[1], (top_features[0], top_features[1])],
#                 kind="average",
#                 ax=ax,
#                 line_kw={"color": "cornflowerblue", "alpha": 0.2}  # Lower opacity for individual lines
#             )
        
#         # Overlay averaged partial dependence for clearer visualization
#         ensemble_avg_display = PartialDependenceDisplay.from_estimator(
#             models[0],
#             X,
#             features=[top_features[0], top_features[1], (top_features[0], top_features[1])],
#             kind="average",
#             ax=ax,
#             line_kw={"color": "darkorange", "linewidth": 2.0}  # Highlight average in darker color
#         )
        
#         # Set title and save
#         fig.suptitle(f"{title}\nFeatures: {top_features[0]} & {top_features[1]}", fontsize=16)
#         plt.savefig(save_path)
#         plt.clf()

#     except TypeError as e:
#         error_message = f"Error in PDP plotting: {str(e)}"
#         with open(os.path.join(save_path, "PDP_error_log.txt"), "w") as f:
#             f.write(error_message)

def plot_ensemble_partial_dependence(models, X, top_features, save_path, title="Ensemble Partial Dependence"):
    """
    Generate partial dependence plots for an ensemble of models on the top two features, 
    including both 1D and 2D partial dependence.

    Args:
        models (list): List of trained models.
        X (pd.DataFrame): DataFrame with features for plotting PDP.
        top_features (list): List of top two features for PDP.
        save_path (str): Path to save the plot.
        title (str): Title for the plot.
    """
    
    try:
        # Create the figure with subplots (1D for each feature, 2D for the interaction)
        fig, axes = plt.subplots(ncols=3, figsize=(12, 5), constrained_layout=True)
        
        # Generate 1D PDPs for each feature in the top two (only ensemble average)
        for i, feature in enumerate(top_features[:2]):
            axes[i].clear()  # Clear the axis to avoid conflicts
            
            # Overlay ensemble average for 1D PDP in darkorange
            disp = PartialDependenceDisplay.from_estimator(
                models[0],  # Use the first model for averaging
                X,
                features=[feature],
                kind="average",
                ax=axes[i],
                line_kw={"color": "darkorange", "linewidth": 2.0}  # Highlight ensemble average
            )
            axes[i].set_title(f"PDP for {feature}")
        
        # Generate 2D PDP for the interaction between the two top features
        axes[2].clear()  # Clear the 2D plot axis
        disp = PartialDependenceDisplay.from_estimator(
            models[0],  # Use the first model for averaging
            X,
            features=[(top_features[0], top_features[1])],
            kind="average",
            ax=axes[2],
            line_kw={"color": "darkorange", "linewidth": 2.0}  # Ensemble average for 2D interaction
        )
        axes[2].set_title(f"2D PDP Interaction: {top_features[0]} & {top_features[1]} (Mean)")
        
        # Set the main title and save the plot
        fig.suptitle(f"{title}\nFeatures: {top_features[0]} & {top_features[1]}", fontsize=16)
        plt.savefig(save_path,  dpi=300, bbox_inches='tight')
        plt.clf()
        plt.close()

    except TypeError as e:
        # Extract the directory path (removes the file name part)
        directory_path = os.path.dirname(save_path)
        # Log any errors that may occur
        error_message = f"Error in PDP plotting: {str(e)}"
        with open(os.path.join(directory_path, "PDP_error_log.txt"), "w") as f:
            f.write(error_message)


# Create a function to find the nearest original cluster for a new latitude/longitude pair
def assign_to_cluster(new_lat, new_lon, centroids):
    # Calculate the geodesic distance between the new point and each centroid
    distances = []
    for centroid in centroids:
        # Calculate geodesic distance (more accurate for lat/lon data)
        dist = geodesic((new_lat, new_lon), (centroid[0], centroid[1])).kilometers
        distances.append(dist)
    
    # Get the index of the closest original cluster
    closest_cluster = np.argmin(distances)

    return closest_cluster

def calculate_q_statistic(data, y_series, cluster_column):
    # Calculate overall mean
    overall_mean = y_series.mean()
    
    # Group by strata (spatial_cluster)
    grouped = data.groupby(cluster_column)

    # Initialize SSW and SST
    SSW = 0
    # N = len(data)  # Total number of observations
    SST = np.sum((y_series - overall_mean) ** 2)
    
    # Calculate SSW
    for cluster_id, group in grouped:
        N_h = len(group)  # Number of units in stratum h
        stratum_mean = y_series[group.index].mean()  # Mean of the current stratum
        stratum_variance = y_series[group.index].var(ddof=1)  # Sample variance
        
        SSW += (N_h - 1) * stratum_variance

    # Calculate Q-statistic
    q_stat = 1 - (SSW / SST)
    
    return q_stat, SSW, SST

def calculate_and_inverse_transform_kde(data, transform=True, min_val=1, max_val=10):
    """
    Calculate the KDE of the input data array and apply an inverse transformation.
    
    Parameters:
        data (np.ndarray): The input array containing the values.
        transform (bool): Perform a log transform on the data.
        min_val (int): The minimum sample weight value
        max_val (int): The maximum sample weight value

        
    Returns:
        original_data (np.ndarray): The original input data.
        inverse_kde_values (np.ndarray): The inverse transformed KDE values for each original data point.
        scaled_kde_vales (np.ndarray): The original KDE scaled between min_val and max_val
    """
    # Drop NA values and ensure data is a 1D array
    data = data[np.isfinite(data)]

    if transform:
        # data = np.log1p(np.log1p(data)) # too extreme...
        data = np.log1p(data)

    
    # Calculate KDE using scipy
    kde = gaussian_kde(data)
    
    # Evaluate the KDE at the original data points
    kde_values = kde(data)
    
    # Scale the KDE values from 1 to 10
    min_kde = np.min(kde_values)
    max_kde = np.max(kde_values)

    scaled_kde_values = min_val + (kde_values - min_kde) * (max_val - min_val) / (max_kde - min_kde)
    
    # Apply inverse transformation
    inverse_kde_values = max_val - (scaled_kde_values - min_val) * (max_val - min_val) / (max_val - min_val)  # Simplified to: 10 - (scaled_kde_values - 1)

    return data, inverse_kde_values, scaled_kde_values

def calculate_morans_I(df, residuals, threshold=1):
    # Pulling the LAT and LON values corresponding to the indices in x_test
    coordinates = list(zip(df['Latitude_0'], df['Longitude_0']))

    # Create a spatial weights matrix using DistanceBand (can be adjusted to other weight schemes)
    w = DistanceBand(coordinates, threshold=threshold, binary=True, p=2)

    # Normalize the spatial weights (optional, but common)
    w.transform = 'R'

    # Calculate Moran's I on the residuals
    moran = Moran(residuals, w)

    # Return the Moran's I value and p-value
    return round(moran.I, 3), moran.p_sim

def spatial_kfold_split(data, coordinates, n_splits=5):
    # Apply KMeans clustering to create spatially-based folds
    kmeans = KMeans(n_clusters=n_splits, random_state=42)
    data['kfold_cluster'] = kmeans.fit_predict(coordinates)
    return data

# === Plot trajectories with 180° crossing fix ===
def split_trajectory(lats, lons):
    """Split trajectory if crossing the dateline to avoid horizontal streaks."""
    segments = []
    current_segment = []
    for i in range(len(lats) - 1):
        current_segment.append((lons[i], lats[i]))
        
        # Check for longitude discontinuity (crossing 180° line)
        if abs(lons[i+1] - lons[i]) > 180:
            segments.append(np.array(current_segment))  # Close current segment
            current_segment = []  # Start new segment
    
    current_segment.append((lons[-1], lats[-1]))
    segments.append(np.array(current_segment))  # Add last segment
    return segments

In [ ]:
def calc_stats(data):
    # Calculate lat/lon interaction and differences
    data = data.assign(
        lat_lon_interaction_0=data['Longitude_0'] * data['Latitude_0'],
        lat_diff=data['Latitude_0'] - data['Latitude_-24'],
        lon_diff=data['Longitude_0'] - data['Longitude_-24'],
        straight_line_efficiency_ratio=data['Dist_from_origin_-24'] / data['Cumulative_Dist_-24']
    )

    # Dew Point and LCL Height calculation
    relative_humidity_values = data[[f'Relative_Humidity_{i}' for i in range(-24, 1)]].values
    temperature_values = data[[f'Temperature_C_{i}' for i in range(-24, 1)]].values
    dew_point_values = temperature_values - ((100 - relative_humidity_values) / 5)
    for i, timestep in enumerate(range(-24, 1)):
        data[f'dew_point_{timestep}'] = dew_point_values[:, i]

    # Calculate LCL from dew point and temperature
    lcl_values = (temperature_values - dew_point_values) / 8 * 1000
    for i, timestep in enumerate(range(-24, 1)):
        data[f'LCL_{timestep}'] = lcl_values[:, i]

    moisture_flux_values = data[[f'Moisture_Flux_{i}' for i in range(-24, 1)]].values

    # Rainfall Estimation based on Relative Humidity and Moisture Flux
    rh_threshold = 80
    rainfall_estimation = moisture_flux_values * (relative_humidity_values / 100)
    rainfall_estimation[relative_humidity_values <= rh_threshold] = 0
    for i, timestep in enumerate(range(-24, 1)):
        data[f'Rainfall_Estimate_{timestep}'] = rainfall_estimation[:, i]

    # Add extra features to main data
    # Step 1: Convert DateTime_0 to datetime format
    for i in range(-24, 1):
        data[f'DateTime_{i}'] = pd.to_datetime(data[f'DateTime_{i}'], format="%m/%d/%Y %H:%M")

        # Step 2: Define a fixed start time (January 1, 1900)
        start_time = pd.Timestamp("1900-01-01 00:00:00")
        # Step 3: Calculate the time difference from this fixed start time
        data[f'time_difference_{i}'] = data[f'DateTime_{i}'] - start_time  # Calculate the difference

        # Step 4: Convert the difference to total seconds
        data[f'time_difference_seconds_{i}'] = data[f'time_difference_{i}'].dt.total_seconds()

        # Step 2: Extract the hour and encode it as cyclical features
        data[f'hour_sin_{i}'] = np.sin(2 * np.pi * data[f'DateTime_{i}'].dt.hour / 24)
        data[f'hour_cos_{i}'] = np.cos(2 * np.pi * data[f'DateTime_{i}'].dt.hour / 24)

        # data['month'] = data['DateTime_0'].dt.month
        # Step 2: Extract the hour and encode it as cyclical features
        data[f'month_sin_{i}'] = np.sin(2 * np.pi * data[f'DateTime_{i}'].dt.month / 12)
        data[f'month_cos_{i}'] = np.cos(2 * np.pi * data[f'DateTime_{i}'].dt.month / 12)

        # Summary Statistics for each field
    fields_to_summarize = [
        'Pressure', 'Potential_Temperature', 'Relative_Humidity', 'Specific_Humidity',
        'Solar_Radiation', 'Mixing_Depth', 'Moisture_Flux', 'Temperature_C', 'Rainfall_Estimate',
        'LCL', 'dew_point', 'Latitude', 'Longitude', 'AltP_meters'
    ]
    for field in fields_to_summarize:
        print(f'Calculating stats for {field}...')
        cols = [f'{field}_{i}' for i in range(-24, 0)]
        data[f'std_{field.lower()}'] = data[cols].std(axis=1)
        data[f'delta_{field.lower()}'] = data[f'{field}_-1'] - data[f'{field}_-24']
        data[f'mean_{field.lower()}'] = data[cols].mean(axis=1)
        data[f'sum_{field.lower()}'] = data[cols].sum(axis=1)

    return data

# Define a function to calculate the radius of curvature
def calc_radius_of_curvature(bearing_current, bearing_next, ptp_dist):
    """
    Calculate the radius of curvature given the current and next bearings (in radians) and the cumulative distance (in meters).
    Bearings should be in radians, and cumulative_dist in meters.
    """ 
    # # Convert cumulative distance from meters to kilometers
    # cumulative_dist = cumulative_dist / 1000  # Meters to kilometers

    # Calculate the change in bearing, ensuring the result is between -pi and pi
    delta_theta = (bearing_next - bearing_current + np.pi) % (2 * np.pi) - np.pi

    # Prevent division by zero by checking if delta_theta is 0
    if delta_theta == 0:
        return np.inf  # Infinite radius, straight line

    # Radius of curvature (arc length / change in angle)
    radius = ptp_dist / delta_theta

    return radius


# Define a function to calculate angular velocity
def calc_angular_velocity(bearing_current, bearing_next, delta_t=1):
    """
    Calculate the angular velocity given the current and next bearings and the time interval.
    Bearings should be in radians and delta_t in hours.
    """
    # Calculate the change in bearing, ensuring the result is between -pi and pi
    delta_theta = (bearing_next - bearing_current + np.pi) % (2 * np.pi) - np.pi
    
    # Angular velocity (radians per hour)
    w = delta_theta / delta_t
    
    return w

R_d = 287.05  # Specific gas constant for dry air in J/(kg·K)
def saturation_vapor_pressure(temperature):
    """
    Calculate saturation vapor pressure using the Tetens formula.
    Temperature in Celsius.
    """
    return 6.112 * np.exp((17.67 * temperature) / (temperature + 243.5))

def clausis_clapeyron(temperature):
    """
    Calculate saturation vapor pressure using Clausis-Clapeyron equation.
    Temperature in Celcius

    example call: clausis_clapeyron(293.15)
    """
    temperature += 273.15

    return 611 * np.exp((((2.5 * 10**6)/461.5) * ((1/273.15) - (1/temperature))))

# Function to calculate moist air density
def moist_air_density(temperature, relative_humidity, pressure):
    """
    Calculate the density of moist air.
    
    Parameters:
    - temperature: Temperature in Celsius
    - relative_humidity: Relative humidity as a percentage (0-100)
    - pressure: Total atmospheric pressure in hPa
    
    Returns:
    - The density of moist air in kg/m^3.
    """
    
    # Convert Pressure to hPa
    pressure = pressure * 100
    
    # Calculate saturation vapor pressure using Clausius-Clapeyron
    e_s = clausis_clapeyron(temperature)
    
    # Calculate actual vapor pressure using relative humidity
    p_v = (relative_humidity / 100.0) * e_s
    
    # Calculate the partial pressure of dry air
    p_d = pressure - p_v
    
    # Constants
    R_d = 287.05  # Specific gas constant for dry air (J/kg·K)
    R_v = 461.5   # Specific gas constant for water vapor (J/kg·K)

    # Convert temperature from Celsius to Kelvin
    temp_k = temperature + 273.15
    
    # Calculate the density of moist air
    density = (p_d / (R_d * temp_k)) + (p_v / (R_v * temp_k))
    
    return density


def mass_of_moist_particle(T, P, RH):
    """
    Calculate the mass of a moist atmospheric particle given temperature, pressure, and humidity.
    
    Parameters:
    T : float
        Temperature in degrees Celsius.
    P : float
        Total pressure in hPa.
    RH : float
        Relative humidity as a fraction (0 to 1).
        
    Returns:
    float
        Mass of a moist particle in kg.
    """
    # Calculate saturation vapor pressure
    P_sat = clausis_clapeyron(T)  # in hPa

    RH = RH / 100
    
    # Calculate partial pressures
    P_water = RH * P_sat  # Partial pressure of water vapor in hPa
    P_dry = P - P_water   # Partial pressure of dry air in hPa
    
    # Molar masses (in g/mol)
    M_d = 28.94  # Molar mass of dry air
    M_w = 18.02  # Molar mass of water vapor
    
    # Calculate average molar mass of moist air (in g/mol)
    M_avg = (M_d * P_dry + M_w * P_water) / P  # in g/mol
    
    # Convert to kg/mol
    M_avg_kg = M_avg / 1000  # Convert g/mol to kg/mol
    
    # Calculate mass of a single particle (in kg)
    N_A = 6.022e23  # Avogadro's number
    m_particle = M_avg_kg / N_A  # in kg
    
    return m_particle


# Define a function to calculate moment of inertia
def calc_moment_of_inertia(radius_of_curvature, temperature, pressure, relative_humidity):
    # mass of an average air particle
    # assign a theoretical mass to the particle
    # mass = 4.81 * 10**-26 # (kg)

    # Moment of inertia
    I = mass_of_moist_particle(temperature, pressure, relative_humidity) * radius_of_curvature**2
    
    return I

# Define a function to calculate angular momentum
def calc_angular_momentum(moment_of_inertia, angular_velocity):
    """
    Calculate the angular momentum given the moment of inertia and angular velocity.
    """
    L = moment_of_inertia * angular_velocity

    return L

def calc_velocity(distance, delta_t=1):
    """Calculate velocity in m/s."""
    return (distance / 1000) / (delta_t / 3600)  # Convert distance from meters to kilometers and delta_t from hours to seconds

def calc_acceleration(velocity_0, velocity_1, delta_t=1):
    return (velocity_1 - velocity_0) / delta_t

omega_earth = 7.2921e-5  # Earth's angular velocity in rad/s
def calc_coriolis_acceleration(lat, velocity):
    """
    Calculate Coriolis acceleration in m/s^2 given latitude and velocity.
    Latitude is in degrees, velocity is in km/h.
    """
    # Convert latitude to radians
    lat_rad = np.deg2rad(lat)

    
    # Calculate Coriolis acceleration
    coriolis_acc = 2 * omega_earth * np.sin(lat_rad) * velocity
    
    return coriolis_acc

def haversine_calc(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])

    # Radius of Earth in meters
    R = 6371000

    # Differences in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c

    return distance  # Distance in meters

def haversine(lat_lon1, lat_lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1 = np.radians(lat_lon1)
    lat2, lon2 = np.radians(lat_lon2)

    # Radius of Earth in meters
    R = 6371000

    # Differences in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c

    return distance

def calculate_wind_speed(lat1, lon1, lat2, lon2, duration_hours=1):
    # Calculate the distance between points in meters
    distance = haversine(lat1, lon1, lat2, lon2)
    
    # Convert hours to seconds
    duration_seconds = duration_hours * 3600
    
    # Calculate wind speed in m/s
    wind_speed = distance / duration_seconds
    
    return wind_speed

def calc_rossby_number(acceleration, wind_speed, lat):
    """
    Calculate the Rossby number given velocity, radius of curvature, and latitude.
    """

    # Coriolis parameter f
    f = 2 * omega_earth * np.sin(np.deg2rad(lat))
    
    if f == 0 or wind_speed == 0:
        return np.inf  # Infinite Rossby number for zero Coriolis effect or zero radius
    
    Ro = acceleration / (f * wind_speed)
    return Ro


g = 9.81  # gravitational acceleration in m/s^2
def calc_brunt_vaisala(pot_temp1, pot_temp2, height1, height2):
    """
    Calculate the Brunt-Väisälä frequency given two points with potential temperatures and heights.
    """
    # Calculate the change in potential temperature and height
    delta_theta = pot_temp2 - pot_temp1
    delta_z = height2 - height1  # Height difference in meters
    
    # Avoid division by zero
    if delta_z == 0 or pot_temp1 == 0:
        return 0
    
    # Calculate N^2 (in s^-2)
    N_squared = (g / pot_temp1) * (delta_theta / delta_z)
    
    # Return N (Brunt-Väisälä frequency)
    return np.sqrt(N_squared)


def calc_divergence(lat1, lon1, lat2, lon2, v1, v2):
    """
    Calculate divergence between two points using the velocity change and distance between them.
    
    Parameters:
    lat1, lon1 : float
        Latitude and longitude of the first point.
    lat2, lon2 : float
        Latitude and longitude of the second point.
    v1, v2 : float
        Velocity at the first and second points in meters per second (m/s).
        
    Returns:
    float
        Divergence in s^{-1}. Returns NaN if any input is NaN or if the distance is zero.
    """
    # Check for NaN values
    if np.isnan(lat1) or np.isnan(lon1) or np.isnan(lat2) or np.isnan(lon2) or np.isnan(v1) or np.isnan(v2):
        return np.nan  # or return 0 if you prefer a specific value
    
    # Calculate distance between two points (in meters)
    dist = geodesic((lat1, lon1), (lat2, lon2)).meters  # Change to meters
    
    # Avoid division by zero
    if dist == 0:
        return 0
    
    # Divergence = (velocity change) / (distance between points)
    divergence = (v2 - v1) / dist  # This is in s^{-1}

    return divergence

def dynamic_viscosity_air(temperature_c, relative_humidity):
    # Convert temperature to Kelvin
    temperature_k = temperature_c + 273.15

    # Sutherland's constant for air
    mu_dry = 1.78e-5  # Dynamic viscosity of dry air at 0°C in Pa·s
    T0 = 273.15  # Reference temperature in K
    S = 111  # Sutherland's constant for air

    # Calculate dynamic viscosity of dry air using Sutherland's formula
    mu_dry_temp = mu_dry * ((T0 + S) / (temperature_k + S)) * (temperature_k / T0) ** (3/2)

    # Empirical relationship for effect of humidity (approx)
    k = 0.02  # Example constant, adjust based on specific conditions
    mu_moist = mu_dry_temp * (1 + k * relative_humidity)

    return mu_moist

def calculate_reynolds_number(pressure, temperature_c, relative_humidity, distance=1):
    """Calculate Reynolds number using input parameters."""
    temperature_k = temperature_c + 273.15  # Convert °C to K

    # Calculate density of moist air kg/m^3
    density = moist_air_density(pressure, temperature_k, relative_humidity)

    # Calculate dynamic viscosity of moist air
    viscosity = dynamic_viscosity_air(temperature_c, relative_humidity)

    # Calculate velocity (ensure this is in m/s)
    velocity = calc_velocity(distance)  # Ensure distance is in meters

    # Calculate Reynolds number
    reynolds_number = (density * velocity * distance) / viscosity
    return reynolds_number

def calc_physics_fields(data: pd.DataFrame) -> pd.DataFrame:
    # # Calculate Radius of Curvature
    # for i in range(-24, 0):
    #     bearings_current_col = f'bearings_ptp_{i}'
    #     bearings_next_col = f'bearings_ptp_{i + 1}'
    #     cumulative_dist_col = f'Distance_ptp_{i}'
    #     data[f'Radius_of_Curvature_{i}'] = data.apply(
    #         lambda row: calc_radius_of_curvature(row[bearings_current_col], row[bearings_next_col], row[cumulative_dist_col]), axis=1)

    # # Calculate Angular Velocity
    # for i in range(-24, 0):
    #     bearings_current_col = f'bearings_ptp_{i}'
    #     bearings_next_col = f'bearings_ptp_{i + 1}'
    #     data[f'Angular_Velocity_{i}'] = data.apply(
    #         lambda row: calc_angular_velocity(row[bearings_current_col], row[bearings_next_col]), axis=1)

    # # Calculate Moment of Inertia
    # for i in range(-24, 0):
    #     pressure_col = f'Pressure_{i}'
    #     temperature_col = f'Temperature_C_{i}'
    #     rh_col = f'Relative_Humidity_{i}'
    #     radius_col = f'Radius_of_Curvature_{i}'
    #     data[f'Moment_of_Inertia_{i}'] = data.apply(
    #         lambda row: calc_moment_of_inertia(row[radius_col], row[pressure_col], row[temperature_col], row[rh_col]), axis=1)

    # # Calculate Angular Momentum
    # for i in range(-24, 0):
    #     angular_velocity_current_col = f'Angular_Velocity_{i}'
    #     moment_of_inertia_current_col = f'Moment_of_Inertia_{i}'
    #     data[f'Angular_Momentum_{i}'] = data.apply(
    #         lambda row: calc_angular_momentum(row[moment_of_inertia_current_col], row[angular_velocity_current_col]), axis=1)

    # Calculate Velocity
    print("Calculating Velocity")
    for i in range(-24, 0):
        distance_col = f'Distance_ptp_{i}'
        data[f'Velocity_{i}'] = data[distance_col].apply(calc_velocity)
    
    # print("Calculating Acceleration...")
    # # Loop over the time steps and calculate velocity
    # for i in range(-24, -1):  # From -24 to -1
    #     velocity_col = f'Velocity_{i}'
    #     next_velocity_col = f'Velocity_{i+1}'
    #     data[f'Acceleration_{i}'] = data.apply(lambda row: calc_acceleration(row[velocity_col],
    #                                                                 row[next_velocity_col]), axis=1)
    #     # print(data[f'Velocity_{i}'])
    # print("Done!")


    # print("Calculating Coriolis Acceleration...")
    # # Loop through time steps and calculate Coriolis acceleration
    # for i in range(-24, 0):
    #     lat_col = f'Latitude_{i}'  # Assuming a column that contains the latitude
    #     velocity_col = f'Velocity_{i}'
        
    #     data[f'Coriolis_Accel_{i}'] = data.apply(lambda row: calc_coriolis_acceleration(row[lat_col], 
    #                                                                                 row[velocity_col]), axis=1)
    # print("Done!")

    # # Example loop to calculate wind speed at each time step
    # print("Calculating Wind Speed...")
    # for i in range(-24, -1):
    #     # Define column names for latitude and longitude at time i and i+1
    #     lat_col_1 = f'Latitude_{i}'
    #     lon_col_1 = f'Longitude_{i}'
    #     lat_col_2 = f'Latitude_{i+1}'
    #     lon_col_2 = f'Longitude_{i+1}'
        
    #     # Calculate distance and then speed
    #     data[f'Wind_Speed_{i}'] = data.apply(
    #         lambda row: haversine(row[lat_col_1], row[lon_col_1], row[lat_col_2], row[lon_col_2]) / 3600
    #         if pd.notnull(row[lat_col_1]) and pd.notnull(row[lon_col_1]) and
    #         pd.notnull(row[lat_col_2]) and pd.notnull(row[lon_col_2])
    #         else np.nan,
    #         axis=1
    #     )

    # print("Done!")
                        
    # print("Calculating Rossby Number...")
    # # Loop through the time steps to calculate Rossby Number
    # for i in range(-24, 0):
    #     lat_col = f'Latitude_{i}'  # Assuming a column for latitude
    #     acceleration_col = f'Acceleration_{i}'
    #     windspeed_col = f'Wind_Speed_{i}'  # From earlier
        
    #     data[f'Rossby_Number_{i}'] = data.apply(lambda row: calc_rossby_number(row[velocity_col],
    #                                                                         row[radius_col],
    #                                                                         row[lat_col]), axis=1)
    # print("Done!")

    # # Calculate Brunt-Väisälä Frequency
    # print("Calculating BV")
    # for i in range(-24, 0):
    #     pot_temp1_col = f'Potential_Temperature_{i}'
    #     pot_temp2_col = f'Potential_Temperature_{i + 1}'
    #     height1_col = f'AltP_meters_{i}'
    #     height2_col = f'AltP_meters_{i + 1}'
    #     data[f'Brunt_Vaisala_Freq_{i}'] = data.apply(
    #         lambda row: calc_brunt_vaisala(row[pot_temp1_col], row[pot_temp2_col], row[height1_col], row[height2_col]), axis=1)

    # print("Calculating div")
    # # Calculate Divergence
    # for i in range(-24, -1):
    #     lat1_col = f'Latitude_{i}'
    #     lon1_col = f'Longitude_{i}'
    #     lat2_col = f'Latitude_{i + 1}'
    #     lon2_col = f'Longitude_{i + 1}'
    #     v1_col = f'Velocity_{i}'
    #     v2_col = f'Velocity_{i + 1}'
    #     data[f'Divergence_{i}'] = data.apply(
    #         lambda row: calc_divergence(row[lat1_col], row[lon1_col], row[lat2_col], row[lon2_col], row[v1_col], row[v2_col]), axis=1)

    # # Calculate Moist Air Density
    # for i in range(-24, 0):
    #     pressure_col = f'Pressure_{i}'
    #     temperature_col = f'Temperature_C_{i}'
    #     rh_col = f'Relative_Humidity_{i}'
    #     data[f'Density_Moist_{i}'] = data.apply(
    #         lambda row: moist_air_density(row[temperature_col], row[rh_col], row[pressure_col]), axis=1)

    # # Calculate Reynolds Number
    # for i in range(-24, 0):
    #     pressure_col = f'Pressure_{i}'
    #     temperature_col = f'Temperature_C_{i}'
    #     rh_col = f'Relative_Humidity_{i}'
    #     data[f'Reynolds_Number_{i}'] = data.apply(
    #         lambda row: calculate_reynolds_number(row[pressure_col], row[temperature_col], row[rh_col]), axis=1)

    # Summary Statistics for each field
    fields_to_summarize = [
        'Velocity', #'Divergence'
        # 'Radius_of_Curvature', 'Angular_Velocity', 'Angular_Momentum', 'Velocity', 'Coriolis_Accel',
        # 'Rossby_Number', 'Brunt_Vaisala_Freq', 'Divergence', 'Density_Moist', 'Reynolds_Number'
    ]
    for field in fields_to_summarize:
        cols = [f'{field}_{i}' for i in range(-24, 0)]
        if field == "Divergence":
            cols = [f'{field}_{i}' for i in range(-24, -1)]
        data[f'std_{field.lower()}'] = data[cols].std(axis=1)
        if field == "Divergence":
            data[f'delta_{field.lower()}'] = data[f'{field}_-2'] - data[f'{field}_-24']
        else:
            data[f'delta_{field.lower()}'] = data[f'{field}_-1'] - data[f'{field}_-24']
        data[f'mean_{field.lower()}'] = data[cols].mean(axis=1)

    return data

def get_key_by_value(dictionary, value):
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
# TODO: Rate Constants? Would this be contradicting the work? 
# Perhaps we are saying right now that all rate constants are affected by meteorology?

# Adding in OH, OH reactivity, how could we do this? Only a small percentage of our data has these measurements...
# - Krigg and treat as a static feature in space?

In [ ]:
# Current PASTEL version:
version_str = "0_1_5"

# Read in CSV file with all of the data we need. Meteorology variables + Pathdata + VOC data
file_path = rf"C:\Users\vwgei\Documents\PVOCAL\data\v{version_str}\v{version_str}_Awakens.csv"

# Load data from CSV into a pandas DataFrame.
df = pd.read_csv(file_path)

# file_path_oc = r"C:\Users\vwgei\Documents\PVOCAL\data\V4\GDAS_A_NEW_HOPE.csv"

# data_oc = pd.read_csv(file_path_oc)

# Replace WAS nodata value with np.nan for consistency
df.replace(-999999.0, np.nan, inplace=True)
df.replace(-888888.0, np.nan, inplace=True)
df.replace(-888888888.0, np.nan, inplace=True)
df.replace(-999999999.0, np.nan, inplace=True)
df.replace(-99999.000000, np.nan, inplace=True)
df.replace(-999.990000, np.nan, inplace=True)
df.replace(-9.999999e+09, np.nan, inplace=True)
df.replace(-9.999990e+08, np.nan, inplace=True)
df.replace(-8.888888e+06, np.nan, inplace=True)
df.replace(-9.999999e+06, np.nan, inplace=True)

# Replace WAS nodata value with np.nan for consistency
df.replace(-8888, np.nan, inplace=True)
df.replace(-999, np.nan, inplace=True)
df.replace(-888, np.nan, inplace=True)
df.replace(-777, np.nan, inplace=True)
df.replace(-777.770000, np.nan, inplace=True)
df.replace(-888.800000, np.nan, inplace=True)
df.replace(-888.880000, np.nan, inplace=True)
df.replace(10000000, np.nan, inplace=True)

# Drop rows with NaN latitudes and longitudes
df.dropna(subset=['Latitude_-24', 'Longitude_-24'], inplace=True)

# Convert -180 to 180 to 0-360
df['Longitude_0'] = df['Longitude_0'].apply(lambda x: (x + 360) if x < 0 else x)

# Replace some outlire or very uncertain values in our dataset
df.loc[df['DMS'] < 1, 'DMS'] = np.nan # below uncertainty threshold in Whole Air Sampler
df.loc[df['O3'] < 1, 'O3'] = np.nan 
df.loc[df['CH4'] < 1500, 'CH4'] = np.nan
# df.loc[df['O3'] < 10, 'O3'] = np.nan

#------------------------------------------------------------------------------
error_metric_columns = ['RMSE', 'R2']

# Initialize a dictionary to store distributions for each error metric
error_distributions = {col: [] for col in error_metric_columns}

#------------------------------------------------------------------------------

# # reduce size of dataframe for hyperparameter testing purposes?
# df, _ = train_test_split(
#     df,
#     test_size=0.9,  # Retain 10% of the data
#     stratify=df['domain_indicator'],
#     random_state=42  # Ensures reproducibility
# )

In [ ]:
df['DateTime_0']

In [ ]:
df = calc_stats(df)
df = calc_physics_fields(df)

In [ ]:
# Extract the colors from tab20b
tab20_colors = plt.cm.tab20b(np.linspace(0, 1, 20))

# Add a 21st color by interpolating (or define a custom color explicitly)
extra_color = np.array([[0.5, 0.5, 0.5, 1.0]])  # A neutral gray as an example
# extra_color = np.array([[0.0, 1.0, 1.0, 1.0]])  # Cyan: full green, full blue, no red
custom_colors = np.vstack([tab20_colors, extra_color])

# Create a custom ListedColormap
custom_tab20_21 = ListedColormap(custom_colors, name='custom_tab20_21')


# Example dictionary mapping numeric labels to custom string labels
label_mapping = {
    0: "ATom",
    1: "ACCLIP",
    2: "DC3",
    3: "DISCOVER-AQ_FRAPPE",
    4: "FIREX-AQ",
    5: "INTEX-A",
    6: "INTEX-B (C130)",
    7: "INTEX-B (DC8)",
    8: "KORUS-AQ",
    9: "PEM TROPICS A (DC8)",
    10: "PEM TROPICS A (P3)",
    11: "PEM TROPICS B (DC8)",
    12: "PEM TROPICS B (P3)",
    13: "PEM WEST A",
    14: "PEM WEST B",
    15: "SEAC4RS (DC8)",
    16: "TRACE A (DC8)",
    17: "TRACE A (P3)",
    18: "TRACE P (DC8)",
    19: "TRACE P (P3)",
    20: "WINTER",
    # Add more mappings as needed
}


# Replace numeric cluster labels with custom string labels
df['domain_label'] = df['domain_indicator'].map(label_mapping)

# Ensure there are no unmapped values
if df['domain_label'].isnull().any():
    raise ValueError("Some numeric labels do not have a corresponding string label in `label_mapping`.")
    

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# Plot the data
sc = ax.scatter(
    df['Longitude_0'].values, 
    df['Latitude_0'].values, 
    c=df['domain_indicator'].values,  # Use numeric values for consistent coloring
    cmap=custom_tab20_21, 
    s=20, 
    transform=ccrs.PlateCarree()
)


# Update the colorbar to match the new number of labels
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
cbar.set_label("Campaign Name")
cbar.set_ticks(range(len(label_mapping)))
cbar.set_ticklabels([label_mapping[i] for i in range(len(label_mapping))])
cbar.ax.invert_yaxis()  # Flip the colorbar to align labels properly


# Title and labels
ax.set_title("Flight Tracks")

# Save and display
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\domain_map.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Convert longitudes to [-180, 180]
def convert_lon(lons):
    return np.array([(lon - 360 if lon > 180 else lon) for lon in lons])

# Create the plot with Mollweide projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.Mollweide(central_longitude=240)})

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines (no labels with Mollweide)
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")

# Scatter plot (categorical data)
sc = ax.scatter(
    convert_lon(df['Longitude_0'].values),
    df['Latitude_0'].values,
    c=df['domain_indicator'].values,
    cmap=custom_tab20_21,
    s=20,
    transform=ccrs.PlateCarree(),  # data still in lat/lon
)

# Colorbar with campaign labels
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
cbar.set_label("Campaign Name")
cbar.set_ticks(range(len(label_mapping)))
cbar.set_ticklabels([label_mapping[i] for i in range(len(label_mapping))])
cbar.ax.invert_yaxis()  # Flip to match your PlateCarree style

# Title
ax.set_title("Flight Tracks (Mollweide)")

# Save & show
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\domain_map_mollweide.png",
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create a colormap and normalize object
cmap = plt.cm.viridis  # or any other colormap you like, like 'plasma', 'coolwarm', etc.
norm = mcolors.Normalize(vmin=df['AltP_meters_0'].min(), vmax=14000)

# Convert longitudes to [-180, 180] for Mollweide compatibility
def convert_lon(lons):
    return [(lon - 360 if lon > 180 else lon) for lon in lons]

# Create figure and axes
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=210)})

# Plot each trajectory with color mapped to AltP_meters_0
for idx, row in df.iterrows():
    lat_cols = row.filter(regex="Latitude*").values
    lon_cols = row.filter(regex="Longitude*").values

    # Split and plot each segment
    segments = split_trajectory(lat_cols, lon_cols)
    for segment in segments:
        if len(segment) > 1:
            ax.plot(
                segment[:, 0],
                segment[:, 1],
                color=cmap(norm(row['AltP_meters_0'])),  # Map altitude to color
                linewidth=0.8,
                transform=ccrs.PlateCarree(),
            )

# Add coastlines and gridlines
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.OCEAN, facecolor="white")

gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False

# Create a ScalarMappable and add colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Needed for matplotlib < 3.1 to avoid a warning
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.025, pad=0.05, extend='max')
cbar.set_label("Altitude at trajectory tail (m)")

# Title and save
plt.title("All Campaign Trajectories")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\all_traj.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Colormap and normalization
cmap = plt.cm.viridis
norm = mcolors.Normalize(vmin=df['AltP_meters_0'].min(), vmax=14000)

# Convert longitudes to [-180, 180]
def convert_lon(lons):
    return np.array([(lon - 360 if lon > 180 else lon) for lon in lons])

# Figure + Mollweide projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.Mollweide(central_longitude=240)})

# Plot each trajectory
for idx, row in df.iterrows():
    lat_cols = row.filter(regex="Latitude*").values
    lon_cols = convert_lon(row.filter(regex="Longitude*").values)

    # Split and plot each segment
    segments = split_trajectory(lat_cols, lon_cols)
    for segment in segments:
        if len(segment) > 1:
            ax.plot(
                segment[:, 0],  # lon
                segment[:, 1],  # lat
                color=cmap(norm(row['AltP_meters_0'])),
                linewidth=0.8,
                transform=ccrs.PlateCarree(),  # data is still in lat/lon
            )

# Coastlines & features
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.OCEAN, facecolor="white")

# Gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")

# Colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.025, pad=0.05, extend='max')
cbar.set_label("Altitude at trajectory tail (m)")

# Title & save
plt.title("Mollweide Projection")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\all_traj_mollweide.png",
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# df.replace(np.Inf, np.nan, inplace=True)
dms_nd, dms_sampweight, dms_kde = calculate_and_inverse_transform_kde(df['DMS'], transform=True, min_val=1, max_val=2)
ethane_nd, ethane_sampweight, ethane_kde = calculate_and_inverse_transform_kde(df['Ethane'], transform=True, min_val=1, max_val=2)
ozone_nd, ozone_sampweight, ozone_kde = calculate_and_inverse_transform_kde(df['O3'], transform=True, min_val=1, max_val=2)
methane_nd, methane_sampweight, methane_kde = calculate_and_inverse_transform_kde(df['CH4'], transform=True, min_val=1, max_val=2)
CO_nd, CO_sampweight, CO_kde = calculate_and_inverse_transform_kde(df['CO'], transform=True, min_val=1, max_val=2)
ch3br_nd, ch3br_sampweight, ch3br_kde = calculate_and_inverse_transform_kde(df['CH3Br'], transform=True, min_val=1, max_val=2)

# Step 5: Create 2x3 subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()  # Flatten the 2D array of axes for easy indexing

# List of compounds for easy iteration
compounds = [
    (dms_nd, dms_sampweight, dms_kde, 'DMS', 'pptv', 0),
    (ethane_nd, ethane_sampweight, ethane_kde, 'Ethane', 'pptv', 1),
    (ozone_nd, ozone_sampweight, ozone_kde, 'Ozone', 'ppbv', 2),
    (methane_nd, methane_sampweight, methane_kde, 'Methane', 'ppbv', 3),
    (CO_nd, CO_sampweight, CO_kde, 'CO', 'ppbv', 4),
    (ch3br_nd, ch3br_sampweight, ch3br_kde, 'CH3Br', 'pptv', 5)
]

# Plot each compound's data in the subplots
for i, (nd, sampweight, kde, llabel, unit, _) in enumerate(compounds):
    axes[i].scatter(nd, kde, color='gray', alpha=0.5, label='Gaussian KDE of\nOriginal Distribution')
    axes[i].scatter(nd, sampweight, color='blue', alpha=0.5, label='Sample Weight')
    axes[i].set_title(llabel)
    axes[i].set_xlabel(f'Log(1 + {unit})')
    axes[i].set_ylabel('Scaled Value')
    axes[i].legend()
    axes[i].grid()

plt.tight_layout()  # Adjust layout to prevent overlap
plt.savefig(f"C:/Users/vwgei/Documents/PVOCAL/plots/standalone/samp_weights_v{version_str}_scaled_log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# List of columns to plot
columns = ["DMS", "Ethane", "O3", "CH4", "CO", "CH3Br",]
labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

# Define colors for each variable
colors = ['purple', 'orange', 'red', 'blue', 'cyan', 'mediumseagreen',]

units = ['(pptv)', '(pptv)', '(ppbv)', '(ppbv)', '(ppbv)', '(pptv)',]

# Create the subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()  # Flatten the 2D array of axes for easy indexing

# Plot each column in a separate subplot
for i, col in enumerate(columns):
    sns.histplot(data=df, x=col, kde=True, color=colors[i], ax=axes[i], bins=30)
    axes[i].set_title(col)  # Set title for each subplot
    axes[i].set_xlabel(col + units[i])
    axes[i].text(0.9, 0.95, labels[i], transform=axes[i].transAxes,
                fontsize=12, fontweight='bold', va='top', ha='left')

# Adjust layout to avoid overlap
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(f"C:/Users/vwgei/Documents/PVOCAL/plots/standalone/ATS_distributions_v{version_str}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# List of columns to plot
columns = ["DMS", "Ethane", "O3", "CH4", "CO", "CH3Br"]
labels = ['(g)', '(h)', '(i)', '(j)', '(k)', '(l)']


# Define colors for each variable
colors = ['purple', 'orange', 'red', 'blue', 'cyan', 'mediumseagreen',]

# Create the subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()  # Flatten the 2D array of axes for easy indexing

# Plot each column in a separate subplot
for i, col in enumerate(columns):
    log_data = np.log1p(df[col])
    sns.histplot(x=log_data, kde=True, color=colors[i], ax=axes[i], bins=30)
    axes[i].set_title(f'Log-transformed {col}')
    axes[i].text(0.9, 0.95, labels[i], transform=axes[i].transAxes,
                fontsize=12, fontweight='bold', va='top', ha='left')

# Adjust layout to avoid overlap
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(f"C:/Users/vwgei/Documents/PVOCAL/plots/standalone/ATS_distributions_v{version_str}_log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
pressure_columns = [f'Pressure_{i}' for i in range(-24, 1)]
potential_temperature_columns = [f'Potential_Temperature_{i}' for i in range(-24, 1)]
mixing_depth_columns = [f'Mixing_Depth_{i}' for i in range(24, 1)]
relative_humidity_columns = [f'Relative_Humidity_{i}' for i in range(-24, 1)]
specific_humidity_columns = [f'Specific_Humidity_{i}' for i in range(-24, 1)]
terrain_altitude_columns = [f'Terrain_Altitude_{i}' for i in range(-24, 1)]
solar_radiation_columns = [f'Solar_Radiation_{i}' for i in range(-24, 1)]
temperature_columns = [f'Temperature_C_{i}' for i in range(-24, 1)]
longitude_columns = [f'Longitude_{i}' for i in range(-24, 1)]
latitude_columns = [f'Latitude_{i}' for i in range(-24, 1)]
altitude_columns = [f'AltP_meters_{i}' for i in range(-24, 1)]

distance_ptp_columns = [f'Distance_ptp_{i}' for i in range(24,1)]
cumulative_dist_columns = [f'Cumulative_Dist_{i}' for i in range(24, 1)]
distance_fo_columns = [f'Dist_from_origin_{i}' for i in range(-24, 1)]
bearing_fo_columns = [f'bearings_from_origin_{i}' for i in range(-24, 1)]
bearing_ptp_columns = [f'bearings_ptp_{i}' for i in range(-24, 1)]


moisture_flux_columns = [f'Moisture_Flux_{i}' for i in range(-23, 0)]
# Dervied Fields
roc_columns = [f'Radius_of_Curvature_{i}' for i in range(-24, 0)]
ang_vel_columns = [f'Angular_Velocity_{i}' for i in range(-24, 0)]
moi_columns = [f'Moment_of_Inertia_{i}' for i in range(-24, 0)]
ang_momentum_columns = [f'Angular_Momentum_{i}' for i in range(-24, 0)]
vel_columns = [f'Velocity_{i}' for i in range(-24, 0)]
acc_columns = [f'Acceleration_{i}' for i in range(-24, -1)]
corr_acc_columns = [f'Coriolis_Accel_{i}' for i in range(-24, 0)]
rossby_num_columns = [f'Rossby_Number_{i}' for i in range(-24, 0)]
bv_columns = [f'Brunt_Vaisala_Freq_{i}' for i in range(-24, 0)]
div_columns = [f'Divergence_{i}' for i in range(-24, -1)]
moist_den_columns = [f'Density_Moist_{i}' for i in range (-24,0)]
reyn_num_columns = [f'Reynolds_Number_{i}' for i in range (-24,0)]
time_diff_seconds = [f'time_difference_seconds_{i}' for i in range (-24,0)]
hour_sin = [f'hour_sin_{i}' for i in range (-24,0)]
hour_cos = [f'hour_cos_{i}' for i in range (-24,0)]
month_sin = [f'month_sin_{i}' for i in range (-24,0)]
month_cos = [f'month_cos_{i}' for i in range (-24,0)]

lcl_columns = [f'LCL_{i}' for i in range(24, 1)]
rainfall_columns = [f'Rainfall_Estimate_{i}' for i in range(24, 1)]

In [ ]:
# Specify Random State
rs = 187 # 64 - CO, CH4, O3 # 187 - ethane, ch3br # 4 - DMS

In [ ]:
# Code I ran once to create the land/ocean binary classifications 
# - This could be optimized and takes about 1.5 hours

# # Load the shapefile manually
# land_shapefile = R"C:\Users\vwgei\Documents\PVOCAL\data\ne_10m_land\ne_10m_land.shp"  # Update this path
# land_polygons = gpd.read_file(land_shapefile)

# # Function to check if a point is on land
# def is_land(lat, lon):
#     point = Point(lon, lat)  # Create a Shapely Point (longitude first!)
#     return int(land_polygons.contains(point).any())  # 1 if land, 0 if ocean

# # Apply function to your DataFrame
# df["land_ocean_bc"] = df.apply(lambda row: is_land(row["Latitude_-24"], row["Longitude_-24"]), axis=1)

# # Create the plot
# fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
# ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# # Add basemap features
# ax.coastlines(resolution='50m')
# ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
# ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
# ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# # Add gridlines
# gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
# gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data
# sc = ax.scatter(
#     df['Longitude_0'].values, 
#     df['Latitude_0'].values, 
#     c=df['land_ocean_bc'].values,  # Use numeric values for consistent coloring
#     cmap='cool', 
#     s=20, 
#     transform=ccrs.PlateCarree()
# )


# # Update the colorbar to match the new number of labels
# cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
# cbar.set_label("Land = 1, Ocean = 0")

# # Title and labels
# ax.set_title("Land/Ocean Binary Classification for trajectory endpoints")

# # Save and display
# plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\land_ocean_classification_-24.png")
# plt.show()

# land_ocean_bc_-24 = df['land_ocean_bc'].values
# np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\land_ocean_bc_-24.npy", land_ocean_bc_-24)

# 0 if the point is over the ocean and 1 if the point is over land
# Data for the -24 hour trajectory timestamp
# df["land_ocean_bc_n24"] = np.load(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\land_ocean_bc_n24.npy")
# # Data for the 0 hour trajectory timestamp (currently unused)
# df["land_ocean_bc_0"] = np.load(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\land_ocean_bc_0.npy")

In [ ]:
land_koppen_data = np.load(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\land_koppen_data.npy")
print(land_koppen_data.shape)
ocean_koppen_data = np.load(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\ocean_koppen_data.npy")
print(ocean_koppen_data.shape)
land_koppen_ll = np.load(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\land_koppen_ll.npy")
print(land_koppen_ll.shape)
ocean_koppen_ll = np.load(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\ocean_koppen_ll.npy")
print(ocean_koppen_ll.shape)

In [ ]:
# land_koppen_data[land_koppen_data == 0] = ocean_koppen_data[land_koppen_data == 0]
# Create a new array where zeros in land_koppen_data are replaced with values from ocean_koppen_data
combined_koppen_data = np.where(land_koppen_data == 0, ocean_koppen_data, land_koppen_data)
combined_koppen_ll = land_koppen_ll

# Extract the longitude band
longitudes = combined_koppen_ll[:, :, 1]

# Find the column index where longitude is closest to +30 degrees
lon_shift_index = np.argmin(np.abs(longitudes - 30), axis=1)[0]

# Roll both arrays along the longitude axis (axis=1)
rolled_koppen_data = np.roll(combined_koppen_data, -lon_shift_index, axis=1)
rolled_koppen_ll = np.roll(combined_koppen_ll, -lon_shift_index, axis=1)
np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\rolled_koppen_data_combined.npy", rolled_koppen_data)
np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\rolled_koppen_ll.npy", rolled_koppen_ll)

In [ ]:
# Build a KDTree for the combined Koppen classification arrays
def build_koppen_lookup(koppen_data, koppen_ll):
    latitudes = koppen_ll[:, :, 0].flatten()
    longitudes = koppen_ll[:, :, 1].flatten()
    koppen_flat = koppen_data.flatten()
    tree = cKDTree(np.column_stack((latitudes, longitudes)))  # Build tree for fast lookup
    return tree, latitudes, longitudes, koppen_flat

# Precompute lookup for land and ocean classifications (combined into one for the future use)
combined_tree, combined_latitudes, combined_longitudes, combined_koppen_flat = build_koppen_lookup(rolled_koppen_data, rolled_koppen_ll)

# Function to get Koppen class from pre-computed tree
def get_koppen_class(lat, lon):
    _, idx = combined_tree.query([lat, lon])
    return combined_koppen_flat[idx]

# Use apply with the optimized function
df['koppen_cluster'] = df.apply(lambda row: get_koppen_class(
    row['Latitude_-24'], row['Longitude_-24']
), axis=1)

# Create figure
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
# fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
# ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Plot the data
sc = ax.scatter(
    df['Longitude_-24'].values, 
    df['Latitude_-24'].values, 
    c=df['koppen_cluster'].values,
    cmap='gist_rainbow',
    # cmap=land_cmap,
    # norm=land_norm, 
    s=10, 
    transform=ccrs.PlateCarree()
)

# Add coastlines
ax.coastlines()

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# Add colorbars
# plt.colorbar(ocean_plot, ax=ax, orientation='horizontal', label="Ocean Koppen Classification", fraction=0.1)
# plt.colorbar(land_plot, ax=ax, orientation='vertical', label="Land Koppen Classification", fraction=0.025)

tick_positions = np.arange(1, 31) + 0.5  # Shift ticks to the middle of each color band

# Create colorbar with proper labels
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.025)
# cbar.set_ticks(tick_positions) # Match classification values
# cbar.set_ticklabels(koppen_labels)  # Assign appropriate labels
# cbar.ax.tick_params(labelsize=8)  # Adjust font size for readability
# cbar.ax.minorticks_off()

# Title and show plot
plt.title("-24 Hour Trajectory Points Classified by Köppen System over Land and Ocean")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_class_-24.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

In [ ]:
# # Assuming combined_koppen_data is already defined
# unique_values = np.unique(combined_koppen_data)  # Get unique values in the dataset

# labeled_array = np.zeros_like(combined_koppen_data, dtype=int)  # Output array
# region_counter = 1  # Start labeling from 1

# for value in unique_values:
#     if value == 0:  # Skip background if needed
#         continue
    
#     mask = combined_koppen_data == value  # Create a mask for this value
#     labeled_mask, num_features = label(mask)  # Label connected regions in this mask
    
#     # Offset labels to ensure uniqueness across different values
#     labeled_mask[labeled_mask > 0] += region_counter  
#     labeled_array[labeled_mask > 0] = labeled_mask[labeled_mask > 0]
    
#     region_counter = labeled_array.max()  # Update counter to prevent label overlap

# print(f"Number of unique regions: {labeled_array.max()}")

# Assuming combined_koppen_data is already defined
unique_values = np.unique(rolled_koppen_data)  # Get unique values in the dataset

koppen_spatial = np.zeros_like(rolled_koppen_data, dtype=int)  # Output array
region_counter = 1  # Start labeling from 1

# Extend the array horizontally (wrap around at -180 and +180 degrees)
extended_data = np.hstack([rolled_koppen_data[:, -360:], rolled_koppen_data, rolled_koppen_data[:, :360]])

# Output array for extended data
extended_labeled = np.zeros_like(extended_data, dtype=int)

for value in unique_values:
    if value == 0:  # Skip background if needed
        continue
    
    mask = extended_data == value  # Create a mask for this value
    labeled_mask, num_features = label(mask)  # Label connected regions in this mask
    
    # Offset labels to ensure uniqueness across different values
    labeled_mask[labeled_mask > 0] += region_counter  
    extended_labeled[labeled_mask > 0] = labeled_mask[labeled_mask > 0]
    
    region_counter = extended_labeled.max()  # Update counter

# Correctly crop back to original dimensions
koppen_spatial = extended_labeled[:, 360:-360]  # Crop 360 pixels from both sides

print(f"Number of unique regions: {koppen_spatial.max()}")

# Define classification values
land_classes = np.arange(1, 31)  # 1 to 30 for land
ocean_classes = np.array([1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 14, 15, 16, 29, 30])

koppen_labels = [
    "Af", "Am", "Aw", "BWh", "BWk", "BSh", "BSk", "Csa", "Csb", "Csc",
    "Cwa", "Cwb", "Cwc", "Cfa", "Cfb", "Cfc", "Dsa", "Dsb", "Dsc", "Dsd",
    "Dwa", "Dwb", "Dwc", "Dwd", "Dfa", "Dfb", "Dfc", "Dfd", "ET", "EF"
]

# Normalized land colors (0-1 range)
land_colors = {
    1: np.array([0, 0, 255]) / 255,  
    2: np.array([0, 120, 255]) / 255,  
    3: np.array([70, 170, 250]) / 255,  
    4: np.array([255, 0, 0]) / 255,  
    5: np.array([255, 150, 150]) / 255,  
    6: np.array([245, 165, 0]) / 255,  
    7: np.array([255, 220, 100]) / 255,  
    8: np.array([255, 255, 0]) / 255,  
    9: np.array([200, 200, 0]) / 255,  
    10: np.array([150, 150, 0]) / 255,  
    11: np.array([150, 255, 150]) / 255,  
    12: np.array([100, 200, 100]) / 255,  
    13: np.array([50, 150, 50]) / 255,  
    14: np.array([200, 255, 80]) / 255,  
    15: np.array([100, 255, 80]) / 255,  
    16: np.array([50, 200, 0]) / 255,  
    17: np.array([255, 0, 255]) / 255,  
    18: np.array([200, 0, 200]) / 255,  
    19: np.array([150, 50, 150]) / 255,  
    20: np.array([150, 100, 150]) / 255,  
    21: np.array([170, 175, 255]) / 255,  
    22: np.array([90, 120, 220]) / 255,  
    23: np.array([75, 80, 180]) / 255,  
    24: np.array([50, 0, 135]) / 255,  
    25: np.array([0, 255, 255]) / 255,  
    26: np.array([55, 200, 255]) / 255,  
    27: np.array([0, 125, 125]) / 255,  
    28: np.array([0, 70, 95]) / 255,  
    29: np.array([178, 178, 178]) / 255,  
    30: np.array([102, 102, 102]) / 255  
}

# Create colormaps
land_cmap = ListedColormap([land_colors[i] for i in sorted(land_colors.keys())])

# Normalize classification values to match colormap indices
land_norm = BoundaryNorm(np.arange(1, 32), land_cmap.N)

# Mask land values where they are 0
masked_land_koppen = np.ma.masked_where(land_koppen_data == 0, land_koppen_data)

# Create figure
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
# fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
# ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))


# Plot land data on top
land_plot = ax.pcolormesh(rolled_koppen_ll[..., 1], rolled_koppen_ll[..., 0], koppen_spatial,
                        #   cmap=land_cmap, norm=land_norm, shading='auto', transform=ccrs.PlateCarree())
                          cmap='gist_rainbow', shading='auto', transform=ccrs.PlateCarree())

np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\koppen_ccl.npy", koppen_spatial)

# # Plot the data
# sc = ax.scatter(
#     df['Longitude_-24'].values, 
#     df['Latitude_-24'].values, 
#     # c=df['domain_indicator'].values,
#     c='saddlebrown',
#     # cmap=custom_tab20_21, 
#     s=1, 
#     alpha=0.3,
#     transform=ccrs.PlateCarree()
# )

# Add coastlines
ax.coastlines()

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# tick_positions = np.arange(1, 31) + 0.5  # Shift ticks to the middle of each color band

# Create colorbar with proper labels
# cbar = plt.colorbar(land_plot, ax=ax, orientation='vertical', fraction=0.025)
# cbar.set_ticks(tick_positions) # Match classification values
# cbar.set_ticklabels(koppen_labels)  # Assign appropriate labels
# cbar.ax.tick_params(labelsize=8)  # Adjust font size for readability
# cbar.ax.minorticks_off()

# Title and show plot
plt.title("Köppen-Geiger Regions Encoded by Connected-Component Labeling")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_cluster.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Generate 4222 unique random colors
np.random.seed(rs)
num_regions = int(koppen_spatial.max())  # Should be 4222
random_colors = np.random.rand(num_regions, 3)  # RGB values between 0 and 1
random_colors = np.vstack([[0, 0, 0], random_colors])  # Add black for 0/background

# Create a custom colormap
koppen_cmap = ListedColormap(random_colors)

with open(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\koppen_cmap.pkl", "wb") as f:
    pickle.dump(koppen_cmap, f) 


# Create figure
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=210)})

# Plot land data with generated random colormap
land_plot = ax.pcolormesh(
    rolled_koppen_ll[..., 1],
    rolled_koppen_ll[..., 0],
    koppen_spatial,
    cmap=koppen_cmap,
    shading='auto',
    transform=ccrs.PlateCarree(),
    # alpha=0.4
)

# Add coastlines
ax.coastlines()

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look


# Title and show plot
plt.title("Köppen-Geiger Regions Encoded by Connected-Component Labeling")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_ccl_wild.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Build a KDTree for the combined Koppen classification arrays
def build_koppen_lookup(koppen_data, koppen_ll):
    latitudes = koppen_ll[:, :, 0].flatten()
    longitudes = koppen_ll[:, :, 1].flatten()
    koppen_flat = koppen_data.flatten()
    tree = cKDTree(np.column_stack((latitudes, longitudes)))  # Build tree for fast lookup
    return tree, latitudes, longitudes, koppen_flat

# Precompute lookup for land and ocean classifications (combined into one for the future use)
combined_spatial_tree, combined_spatial_latitudes, combined_spatial_longitudes, combined_spatial_koppen_flat = build_koppen_lookup(koppen_spatial, rolled_koppen_ll)

# Function to get Koppen class from pre-computed tree
def get_koppen_spatial_class(lat, lon):
    _, idx = combined_spatial_tree.query([lat, lon])
    return combined_spatial_koppen_flat[idx]

# Use apply with the optimized function
df['koppen_spatial_cluster'] = df.apply(lambda row: get_koppen_spatial_class(
    row['Latitude_-24'], row['Longitude_-24']
), axis=1)


# Create figure
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

# Plot the data
sc = ax.scatter(
    df['Longitude_-24'].values, 
    df['Latitude_-24'].values, 
    c=df['koppen_spatial_cluster'].values,
    cmap=koppen_cmap,
    # cmap=land_cmap,
    # norm=land_norm, 
    s=10, 
    transform=ccrs.PlateCarree()
)

# Add coastlines
ax.coastlines()

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look


# tick_positions = np.arange(1, 31) + 0.5  # Shift ticks to the middle of each color band

# Create colorbar with proper labels
# cbar = plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.025)

# Title and show plot
plt.title("-24 Hour Trajectory Points Classified by CCL Köppen-Geiger Regimes")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_-24_class.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

In [ ]:
# Some of our clusters have very few data points...
for value, count in df['koppen_spatial_cluster'].value_counts().items():
    print(value, count)

In [ ]:
# === Create figure ===
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=210)})

# === Plot the Köppen-Geiger regions as background ===
land_plot = ax.pcolormesh(
    rolled_koppen_ll[..., 1],
    rolled_koppen_ll[..., 0],
    koppen_spatial,
    cmap=koppen_cmap,
    shading='auto',
    alpha=0.2,  # Lower opacity for background
    transform=ccrs.PlateCarree(),
)

# === Add coastlines and gridlines ===
ax.coastlines()
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False

# === Normalize Koppen clusters to colormap range ===
koppen_norm = Normalize(vmin=koppen_spatial.min(), vmax=koppen_spatial.max())

# === Plot each trajectory with wrap-around fix ===
for idx, row in df.iterrows():
    lat_cols = row.filter(regex="Latitude*").values
    lon_cols = row.filter(regex="Longitude*").values
    koppen_cluster = row['koppen_spatial_cluster']  # Corrected column usage

    # Normalize and get trajectory color
    trajectory_color = koppen_cmap(koppen_norm(koppen_cluster)) if 0 <= koppen_cluster <= koppen_spatial.max() else [0, 0, 0]

    # Split and plot each segment
    segments = split_trajectory(lat_cols, lon_cols)
    for segment in segments:
        if len(segment) > 1:  # Skip empty segments
            ax.plot(
                segment[:, 0],
                segment[:, 1],
                color=trajectory_color,
                linewidth=0.8,
                transform=ccrs.PlateCarree(),
            )

# # === Optional: Add legend (optional if too cluttered with 10+ entries) ===
# if df.shape[0] <= 10:  # Only show legend if few trajectories to avoid clutter
#     ax.legend(loc='upper right', fontsize=8)

# # === Title and Save ===
plt.title("Köppen-Geiger CCL Clusters with PASTEL Trajectories")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_ccl_trajectories.png", dpi=300)
plt.show()

In [ ]:
# Compute centroids of each cluster
def compute_centroids(df, column):
    centroids = {}  # Dictionary to store centroid coordinates for each cluster
    for value in df[column].unique():  # Iterate over unique cluster values
        cluster_points = df[df[column] == value][['Latitude_-24', 'Longitude_-24']]  # Extract cluster points
        centroids[value] = cluster_points.mean().values  # Compute and store mean coordinates
    return centroids

# Identify small clusters based on a threshold
def identify_small_clusters(df, threshold=25):
    cluster_counts = df['koppen_spatial_cluster'].value_counts()  # Count occurrences of each cluster
    small_clusters = cluster_counts[cluster_counts < threshold].index.tolist()  # Identify clusters below threshold
    return small_clusters

# Reclassify small clusters by assigning them to the nearest larger cluster
def reclassify_small_clusters(df, small_clusters, centroids):
    large_clusters = {k: v for k, v in centroids.items() if k not in small_clusters}  # Keep only large clusters
    large_cluster_tree = cKDTree(list(large_clusters.values()))  # Build spatial tree for fast nearest-neighbor search
    
    # Copy original cluster labels
    df['koppen_spatial_cluster_reclass'] = df['koppen_spatial_cluster']
    df['binary_class_koppen'] = 0  # Initialize new column with 0 (not reclassified)
    
    # Dictionary to store mapping of small clusters to their new assignments
    reclassification_mapping = {}

    for cluster in small_clusters:
        # Get points belonging to the small cluster
        small_cluster_points = df[df['koppen_spatial_cluster'] == cluster][['Latitude_-24', 'Longitude_-24']].values
        
        if len(small_cluster_points) == 0:
            continue  # Skip empty clusters
        
        # Find the nearest large cluster using spatial tree
        _, nearest_idx = large_cluster_tree.query(small_cluster_points)
        nearest_large_cluster = list(large_clusters.keys())[nearest_idx[0]]
        
        # Update cluster labels and mark reclassified points
        df.loc[df['koppen_spatial_cluster'] == cluster, 'koppen_spatial_cluster_reclass'] = nearest_large_cluster
        df.loc[df['koppen_spatial_cluster'] == cluster, 'binary_class_koppen'] = 1  # Mark as reclassified
        
        # Store in the mapping dictionary
        reclassification_mapping[cluster] = nearest_large_cluster
    
    return df, reclassification_mapping

# Compute centroids for initial clusters
centroids = compute_centroids(df, 'koppen_spatial_cluster')
# Identify small clusters
small_clusters = identify_small_clusters(df, threshold=25)
# Reclassify small clusters and update DataFrame
df, small_reclassified_clusters = reclassify_small_clusters(df, small_clusters, centroids)
# Compute centroids for the reclassified clusters
centroids_reclass = compute_centroids(df, 'koppen_spatial_cluster_reclass')

# Create a mapping from old cluster labels to new sequential labels
unique_clusters = sorted(df['koppen_spatial_cluster_reclass'].unique())  # Sorted for consistency
cluster_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_clusters, start=1)}

# Apply mapping to create a new relabeled column
df["koppen_spatial_relabel_reclass"] = df["koppen_spatial_cluster_reclass"].map(cluster_mapping)


# Generate random colors for each unique cluster
np.random.seed(rs)
random_colors = np.random.rand(len(unique_clusters), 3)  # RGB values

# Create a colormap from these random colors
custom_cmap = ListedColormap(random_colors)


# Create figure with PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

# Scatter plot of cluster points
sc = ax.scatter(
    df['Longitude_-24'].values, 
    df['Latitude_-24'].values, 
    c=df["koppen_spatial_relabel_reclass"].values, 
    cmap=custom_cmap, 
    s=10, 
    transform=ccrs.PlateCarree()
)

# Add coastlines and gridlines
ax.coastlines()
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  

# Create colorbar
# cbar = plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.025)
# cbar.set_label("Cluster Labels")d

# Identify and label clusters
unique_clusters = df["koppen_spatial_relabel_reclass"].unique()
for cluster in unique_clusters:
    # Get cluster points
    cluster_points = df[df["koppen_spatial_relabel_reclass"] == cluster][['Longitude_-24', 'Latitude_-24']].values
    if len(cluster_points) > 0:
        # Find approximate midpoint
        mid_idx = len(cluster_points) // 2
        mid_long, mid_lat = cluster_points[mid_idx]

        # Adjust longitude if necessary
        if mid_long > 100:
            mid_long -= 360  

        # Add label to the midpoint of the cluster
        ax.text(mid_long, mid_lat, str(cluster), fontsize=10, ha='center', va='center',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.20), transform=ccrs.PlateCarree())

# Set title and save figure
plt.title("-24 Hour Trajectory Points Labeled by Köppen-Geiger CCL Classification")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_-24_reclass_relabel.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

# Create figure with PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

# Scatter plot of cluster points
sc = ax.scatter(
    df['Longitude_0'].values, 
    df['Latitude_0'].values, 
    c=df["koppen_spatial_relabel_reclass"].values, 
    cmap=custom_cmap, 
    s=10, 
    transform=ccrs.PlateCarree()
)

# Add coastlines and gridlines
ax.coastlines()
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  

# Set title and save figure
plt.title("Time-of-Sampling (hour 0) Trajectory Points\nLabeled by -24 Hour Köppen-Geiger CCL Classification")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_0_reclass_relabel.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

In [ ]:
koppen_spatial_copy = np.copy(koppen_spatial)

# Step 1: Replace mapped values
for key, value in small_reclassified_clusters.items():
    koppen_spatial_copy[koppen_spatial_copy == key] = value

# relabel the actual clusters in koppen_spatial to align with point labels in 'koppen_spatial_cluster_reclass'
max_value = koppen_spatial_copy.max()

# Step 2: Handle unmapped values in 1-82 range
unique_values = np.unique(koppen_spatial_copy)

unmapped_values = [v for v in unique_values if 1 <= v <= len(np.unique(df["koppen_spatial_relabel_reclass"].values)) and v not in cluster_mapping]
# Assign new values starting from max_value + 1
for new_label, old_label in enumerate(unmapped_values, start=max_value + 1):
    koppen_spatial_copy[koppen_spatial_copy == old_label] = new_label

# Step 1: Replace mapped values
for key, value in cluster_mapping.items():
    koppen_spatial_copy[koppen_spatial_copy == key] = value

koppen_spatial_float = koppen_spatial_copy.astype(float)
# Mask out values where koppen_spatial_copy > 82
koppen_spatial_float[koppen_spatial_float > len(np.unique(df["koppen_spatial_relabel_reclass"].values))] = np.nan

np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\koppen_ccl_remapped.npy", koppen_spatial_copy)
np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\koppen_spatial_float.npy", koppen_spatial_float)

# Create figure
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=210)})

# Plot land data with generated random colormap
land_plot = ax.pcolormesh(
    rolled_koppen_ll[..., 1],
    rolled_koppen_ll[..., 0],
    koppen_spatial_float,
    cmap='managua',
    transform=ccrs.PlateCarree(),
)

# Add coastlines
ax.coastlines()

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

cbar = plt.colorbar(land_plot)

# Title and show plot
plt.title("Köppen-Geiger Regions Where We Have Data")
# plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_float.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Function to generate zoomed-in cluster plots with properly positioned labels
def plot_zoomed_clusters(region_name, lon_bounds, lat_bounds, save_path):
    fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

    # Scatter plot of cluster points
    sc = ax.scatter(
        df['Longitude_-24'].values, 
        df['Latitude_-24'].values, 
        c=df['koppen_spatial_relabel_reclass'].values,  
        cmap=custom_cmap, 
        s=10, 
        transform=ccrs.PlateCarree()
    )

    # Set zoomed-in extent
    ax.set_extent(lon_bounds + lat_bounds, crs=ccrs.PlateCarree())

    # Add coastlines and gridlines
    ax.coastlines()
    gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
    gl.top_labels = gl.right_labels = False  

    # Create colorbar
    # cbar = plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.025)
    # cbar.set_label("Cluster Labels (Renumbered)")

    # Label clusters at a representative in-bounds position
    for cluster, new_label in cluster_mapping.items():
        # Get cluster points within the zoomed-in region
        cluster_points = df[df['koppen_spatial_cluster_reclass'] == cluster][['Longitude_-24', 'Latitude_-24']].values
        in_bounds_points = [pt for pt in cluster_points if lon_bounds[0] <= pt[0] <= lon_bounds[1] and lat_bounds[0] <= pt[1] <= lat_bounds[1]]

        if in_bounds_points:
            # Pick a good label position (median of in-bound points)
            in_bounds_points = np.array(in_bounds_points)
            mid_long, mid_lat = np.median(in_bounds_points, axis=0)

            # Adjust longitude if necessary
            if mid_long > 100:
                mid_long -= 360  

            # Add relabeled cluster number
            ax.text(mid_long, mid_lat, str(new_label), fontsize=10, ha='center', va='center',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.3), transform=ccrs.PlateCarree())

    # Set title and save figure
    plt.title(f"Köppen-Geiger CCL Clusters: {region_name}")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.clf()
    plt.close()


# Plot North America
plot_zoomed_clusters(
    "North America", 
    lon_bounds=[-170, -50], 
    lat_bounds=[5, 75], 
    save_path=r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_-24_reclass_NA.png"
)

# Plot East Asia
plot_zoomed_clusters(
    "East Asia", 
    lon_bounds=[90, 150], 
    lat_bounds=[15, 55], 
    save_path=r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_-24_reclass_EastAsia.png"
)

In [ ]:
df['koppen_spatial_cluster_small'] = df['koppen_spatial_cluster'].copy()

In [ ]:
# Compute centroids of each cluster
def compute_centroids_small(df, column):
    centroids = {}  # Dictionary to store centroid coordinates for each cluster
    for value in df[column].unique():  # Iterate over unique cluster values
        cluster_points = df[df[column] == value][['Latitude_-24', 'Longitude_-24']]  # Extract cluster points
        centroids[value] = cluster_points.mean().values  # Compute and store mean coordinates
    return centroids

# Identify small clusters based on a threshold
def identify_small_clusters_small(df, threshold=10):
    cluster_counts = df['koppen_spatial_cluster_small'].value_counts()  # Count occurrences of each cluster
    small_clusters = cluster_counts[cluster_counts < threshold].index.tolist()  # Identify clusters below threshold
    return small_clusters

# Reclassify small clusters by assigning them to the nearest larger cluster
def reclassify_small_clusters_small(df, small_clusters, centroids):
    large_clusters = {k: v for k, v in centroids.items() if k not in small_clusters}  # Keep only large clusters
    large_cluster_tree = cKDTree(list(large_clusters.values()))  # Build spatial tree for fast nearest-neighbor search
    
    # Copy original cluster labels
    df['koppen_spatial_cluster_reclass_small'] = df['koppen_spatial_cluster_small']
    # print(df['koppen_spatial_cluster_reclass_small']) # DEBUG
    df['binary_class_koppen_small'] = 0  # Initialize new column with 0 (not reclassified)
    
    # Dictionary to store mapping of small clusters to their new assignments
    reclassification_mapping = {}

    for cluster in small_clusters:
        # Get points belonging to the small cluster
        small_cluster_points = df[df['koppen_spatial_cluster_small'] == cluster][['Latitude_-24', 'Longitude_-24']].values
        
        if len(small_cluster_points) == 0:
            continue  # Skip empty clusters
        
        # Find the nearest large cluster using spatial tree
        _, nearest_idx = large_cluster_tree.query(small_cluster_points)
        nearest_large_cluster = list(large_clusters.keys())[nearest_idx[0]]
        
        # Update cluster labels and mark reclassified points
        df.loc[df['koppen_spatial_cluster_small'] == cluster, 'koppen_spatial_cluster_reclass_small'] = nearest_large_cluster
        df.loc[df['koppen_spatial_cluster_small'] == cluster, 'binary_class_koppen_small'] = 1  # Mark as reclassified
        
        # Store in the mapping dictionary
        reclassification_mapping[cluster] = nearest_large_cluster
    
    return df, reclassification_mapping

# Compute centroids for initial clusters
centroids_small = compute_centroids_small(df, 'koppen_spatial_cluster_small')
# Identify small clusters
small_clusters_small = identify_small_clusters_small(df, threshold=65)
# Reclassify small clusters and update DataFrame
df, small_reclssified_clusters_small = reclassify_small_clusters_small(df, small_clusters_small, centroids_small)
# Compute centroids for the reclassified clusters
centroids_reclass_small = compute_centroids_small(df, 'koppen_spatial_cluster_reclass_small')

# Create a mapping from old cluster labels to new sequential labels
unique_clusters_small = sorted(df['koppen_spatial_cluster_reclass_small'].unique())  # Sorted for consistency
cluster_mapping_small = {old_label: new_label for new_label, old_label in enumerate(unique_clusters_small, start=1)}

# Apply mapping to create a new relabeled column
df["koppen_spatial_relabel_reclass_small"] = df["koppen_spatial_cluster_reclass_small"].map(cluster_mapping_small)


# Generate random colors for each unique cluster
np.random.seed(rs)
random_colors_small = np.random.rand(len(unique_clusters_small), 3)  # RGB values

# Create a colormap from these random colors
custom_cmap_small = ListedColormap(random_colors_small)


# Create figure with PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

# Scatter plot of cluster points
sc = ax.scatter(
    df['Longitude_-24'].values, 
    df['Latitude_-24'].values, 
    c=df["koppen_spatial_relabel_reclass_small"].values, 
    cmap=custom_cmap_small, 
    s=10, 
    transform=ccrs.PlateCarree()
)

# Add coastlines and gridlines
ax.coastlines()
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  

# Create colorbar
# cbar = plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.025)
# cbar.set_label("Cluster Labels")

# Identify and label clusters
unique_clusters_small = df["koppen_spatial_relabel_reclass_small"].unique()
for cluster in unique_clusters_small:
    # Get cluster points
    cluster_points = df[df["koppen_spatial_relabel_reclass_small"] == cluster][['Longitude_-24', 'Latitude_-24']].values
    if len(cluster_points) > 0:
        # Find approximate midpoint
        mid_idx = len(cluster_points) // 2
        mid_long, mid_lat = cluster_points[mid_idx]

        # Adjust longitude if necessary
        if mid_long > 100:
            mid_long -= 360  

        # Add label to the midpoint of the cluster
        ax.text(mid_long, mid_lat, str(cluster), fontsize=10, ha='center', va='center',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.20), transform=ccrs.PlateCarree())

# Set title and save figure
plt.title("-24 Hour Trajectory Points Labeled by Köppen-Geiger CCL Classification (DMS)")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_-24_reclass_relabel_small.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

# Create figure with PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

# Scatter plot of cluster points
sc = ax.scatter(
    df['Longitude_0'].values, 
    df['Latitude_0'].values, 
    c=df["koppen_spatial_relabel_reclass_small"].values, 
    cmap=custom_cmap_small, 
    s=10, 
    transform=ccrs.PlateCarree()
)

# Add coastlines and gridlines
ax.coastlines()
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  

# Set title and save figure
plt.title("Time-of-Sampling (hour 0) Trajectory Points\nLabeled by -24 Hour Köppen-Geiger CCL Classification (DMS)")
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_spatial_0_reclass_relabel_small.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

In [ ]:
koppen_spatial_copy_small = np.copy(koppen_spatial)

# Step 1: Replace mapped values
for key, value in small_reclssified_clusters_small.items():
    koppen_spatial_copy_small[koppen_spatial_copy_small == key] = value

# relabel the actual clusters in koppen_spatial to align with point labels in 'koppen_spatial_cluster_reclass'
max_value_small = koppen_spatial_copy_small.max()

# Step 2: Handle unmapped values in 1-82 range
unique_values_small = np.unique(koppen_spatial_copy_small)

unmapped_values_small = [v for v in unique_values_small if 1 <= v <= len(np.unique(df["koppen_spatial_relabel_reclass_small"].values)) and v not in cluster_mapping_small]
# Assign new values starting from max_value + 1
for new_label, old_label in enumerate(unmapped_values_small, start=max_value_small + 1):
    koppen_spatial_copy_small[koppen_spatial_copy_small == old_label] = new_label

# Step 1: Replace mapped values
for key, value in cluster_mapping_small.items():
    koppen_spatial_copy_small[koppen_spatial_copy_small == key] = value

koppen_spatial_float_small = koppen_spatial_copy_small.astype(float)

koppen_spatial_float_small[koppen_spatial_float_small > len(np.unique(df["koppen_spatial_relabel_reclass_small"].values))] = np.nan

np.save(r"C:\Users\vwgei\Documents\PVOCAL\data\Koppen\koppen_ccl_remapped_small.npy", koppen_spatial_copy_small)

# Create figure
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=210)})

# Plot land data with generated random colormap
land_plot = ax.pcolormesh(
    rolled_koppen_ll[..., 1],
    rolled_koppen_ll[..., 0],
    koppen_spatial_float_small,
    cmap='viridis',
    transform=ccrs.PlateCarree(),
)

# # Plot land data with generated random colormap
# dms_plot = ax.scatter(
#     df['Longitude_-24'],
#     df['Latitude_-24'],
#     c=df['DMS'],
#     cmap='plasma',
#     transform=ccrs.PlateCarree(),
# )

# Add coastlines
ax.coastlines()

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

cbar = plt.colorbar(land_plot)

# Title and show plot
plt.title("Köppen-Geiger Regions Where We Have Data (DMS)")
# plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\koppen_ccl_wild.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
len(centroids_reclass)

In [ ]:
# After we reclassify our tiny clusters our number of samples per cluster looks much more reasonable
for value, count in df['koppen_spatial_cluster_reclass'].value_counts().items():
    print(value, count)

In [ ]:
# Haversine function
def haversine(lat_lon1, lat_lon2):
    R = 6371  # Radius of Earth in meters
    dlat = lat_lon2[0] - lat_lon1[0]
    dlon = lat_lon2[1] - lat_lon1[1]
    a = np.sin(dlat / 2)**2 + np.cos(lat_lon1[0]) * np.cos(lat_lon2[0]) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Parallelized function to calculate distances from one point to all centroids
def distance_to_centroids(point, centroids):
    return [haversine(point, centroid) for centroid in centroids]

In [ ]:
# Convert lat/lon to radians for both points and centroids
coordinates = df[['Latitude_0', 'Longitude_0']].copy()
coordinates.loc[coordinates['Longitude_0'] > 100, 'Longitude_0'] -= 360
coords_radians = np.radians(coordinates.values)

world_cites = pd.read_csv(r"C:\Users\vwgei\Documents\PVOCAL\data\worldcities.csv")
top_1000 = world_cites.sort_values('population', ascending=False).iloc[:1000]

top_1000_ll = top_1000[['lat', 'lng']]
top_1000_ll.loc[top_1000_ll['lng'] > 100, 'lng'] -= 360
top_1000_radians = np.radians(top_1000_ll.values)

# Run in parallel: calculate distance from each point to all centroids
distances_cities = Parallel(n_jobs=10)(delayed(distance_to_centroids)(coords_radians[i], top_1000_radians) for i in range(len(coords_radians)))
dist_matrix_cities = np.array(distances_cities)

# Optional: Compute minimum and mean distance to centroids as features
df['min_distance_to_t1000_cities'] = dist_matrix_cities.min(axis=1)
df['mean_distance_to_t1000_cities'] = dist_matrix_cities.mean(axis=1)

In [ ]:
df_copy = df[['Latitude_0', 'Longitude_0', 'koppen_spatial_relabel_reclass']].copy()
df_copy.loc[df_copy['Longitude_0'] > 100, 'Longitude_0'] -= 360
sample_points = df_copy.sample(n=5, replace=True, random_state=42)
# Convert sample points to radians (assuming they contain 'lat' and 'lng' columns)
sample_points_ll = sample_points[['Latitude_0', 'Longitude_0']]
sample_points_ll.loc[sample_points_ll['Latitude_0'] > 100, 'Longitude_0'] -= 360
sample_points_radians = np.radians(sample_points_ll.values)

# Calculate distance matrix for each of the 5 sample points to each of the top 100 cities
distances_sample = [
    distance_to_centroids(sample_point, top_1000_radians) 
    for sample_point in sample_points_radians
]

# Convert to numpy array for matrix operations
dist_matrix_sample = np.array(distances_sample)

# Set up the map projection
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=210))
ax.set_global()

# Add coastlines and other map features
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

# Plot all points from df in gray
ax.scatter(
    df['Longitude_0'],
    df['Latitude_0'],
    color='gray',
    s=5,
    alpha=0.5,
    label='All points from input data',
    transform=ccrs.PlateCarree()
)

# Plot top 100 cities as blue dots
ax.scatter(
    top_1000_ll['lng'],
    top_1000_ll['lat'],
    color='blue',
    s=10,
    label='Top 1000 Cities',
    transform=ccrs.PlateCarree()
)

# Plot the 7 sample points as red dots
ax.scatter(
    sample_points_ll['Longitude_0'],
    sample_points_ll['Latitude_0'],
    color='red',
    s=40,
    label='Sample Points',
    transform=ccrs.PlateCarree()
)

# Draw lines connecting each sample point to each top 100 city
for sample_point in sample_points_radians:
    sample_lat, sample_lng = np.degrees(sample_point)
    for city_lat, city_lng in top_1000_ll.values:
        ax.plot(
            [sample_lng, city_lng],
            [sample_lat, city_lat],
            color='green',
            linewidth=0.1,
            linestyle='--',
            transform=ccrs.Geodetic()
        )

# Add a legend and title
plt.legend(loc='lower left')
plt.title('Connections Between Sample Points and Top 1000 Most Populated Cities for 5 Random Samples')
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\top_1000_cities.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Define classification values
land_classes = np.arange(1, 31)  # 1 to 30 for land
ocean_classes = np.array([1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 14, 15, 16, 29, 30])

koppen_labels = [
    "Af", "Am", "Aw", "BWh", "BWk", "BSh", "BSk", "Csa", "Csb", "Csc",
    "Cwa", "Cwb", "Cwc", "Cfa", "Cfb", "Cfc", "Dsa", "Dsb", "Dsc", "Dsd",
    "Dwa", "Dwb", "Dwc", "Dwd", "Dfa", "Dfb", "Dfc", "Dfd", "ET", "EF"
]

# Normalized land colors (0-1 range)
land_colors = {
    1: np.array([0, 0, 255]) / 255,  
    2: np.array([0, 120, 255]) / 255,  
    3: np.array([70, 170, 250]) / 255,  
    4: np.array([255, 0, 0]) / 255,  
    5: np.array([255, 150, 150]) / 255,  
    6: np.array([245, 165, 0]) / 255,  
    7: np.array([255, 220, 100]) / 255,  
    8: np.array([255, 255, 0]) / 255,  
    9: np.array([200, 200, 0]) / 255,  
    10: np.array([150, 150, 0]) / 255,  
    11: np.array([150, 255, 150]) / 255,  
    12: np.array([100, 200, 100]) / 255,  
    13: np.array([50, 150, 50]) / 255,  
    14: np.array([200, 255, 80]) / 255,  
    15: np.array([100, 255, 80]) / 255,  
    16: np.array([50, 200, 0]) / 255,  
    17: np.array([255, 0, 255]) / 255,  
    18: np.array([200, 0, 200]) / 255,  
    19: np.array([150, 50, 150]) / 255,  
    20: np.array([150, 100, 150]) / 255,  
    21: np.array([170, 175, 255]) / 255,  
    22: np.array([90, 120, 220]) / 255,  
    23: np.array([75, 80, 180]) / 255,  
    24: np.array([50, 0, 135]) / 255,  
    25: np.array([0, 255, 255]) / 255,  
    26: np.array([55, 200, 255]) / 255,  
    27: np.array([0, 125, 125]) / 255,  
    28: np.array([0, 70, 95]) / 255,  
    29: np.array([178, 178, 178]) / 255,  
    30: np.array([102, 102, 102]) / 255  
}

# Normalized ocean colors (0-1 range)
ocean_colors = {
    1: np.array([76, 114, 0]) / 255,  
    2: np.array([152, 129, 0]) / 255,  
    3: np.array([205, 245, 122]) / 255,  
    4: np.array([155, 76, 0]) / 255,  
    6: np.array([255, 211, 128]) / 255,  
    8: np.array([0, 115, 76]) / 255,  
    9: np.array([0, 168, 132]) / 255,  
    10: np.array([0, 220, 197]) / 255,  
    11: np.array([68, 78, 139]) / 255,  
    12: np.array([123, 142, 245]) / 255,  
    14: np.array([115, 0, 0]) / 255,  
    15: np.array([168, 0, 0]) / 255,  
    16: np.array([245, 123, 122]) / 255,  
    29: np.array([199, 0, 255]) / 255,  
    30: np.array([77, 0, 116]) / 255  
}

# Create colormaps
land_cmap = ListedColormap([land_colors[i] for i in sorted(land_colors.keys())])
ocean_cmap = ListedColormap([ocean_colors[i] for i in sorted(ocean_colors.keys())])

# Normalize classification values to match colormap indices
land_norm = BoundaryNorm(np.arange(1, 32), land_cmap.N)
ocean_norm = BoundaryNorm(ocean_classes, ocean_cmap.N)

# Reverse colormap and normalization
reversed_land_cmap = ListedColormap(land_cmap.colors[::-1])  # Reverse colors
reversed_norm = BoundaryNorm(np.arange(1, 32), reversed_land_cmap.N)

# Mask land values where they are 0
masked_land_koppen = np.ma.masked_where(land_koppen_data == 0, land_koppen_data)

# Set your desired alpha
flat_alpha = 0.2

# Add alpha to land colors
land_colors_rgba = [np.append(color, flat_alpha) for color in [land_colors[i] for i in sorted(land_colors.keys())]]
ocean_colors_rgba = [np.append(color, flat_alpha) for color in [ocean_colors[i] for i in sorted(ocean_colors.keys())]]

# Re-create colormaps with RGBA
land_cmap = ListedColormap(land_colors_rgba)
ocean_cmap = ListedColormap(ocean_colors_rgba)

# Combined plot?
# Create figure and axes
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=210)})
# ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Plot ocean data first
ocean_plot = ax.pcolormesh(ocean_koppen_ll[..., 1], ocean_koppen_ll[..., 0], ocean_koppen_data,
                            # cmap=ocean_cmap, norm=ocean_norm, shading='auto', transform=ccrs.PlateCarree())
                            cmap=land_cmap, norm=land_norm, shading='auto', transform=ccrs.PlateCarree(),
                            zorder=2)


# Plot land data on top
land_plot = ax.pcolormesh(land_koppen_ll[..., 1], land_koppen_ll[..., 0], land_koppen_data,
                          cmap=land_cmap, norm=land_norm, shading='auto', transform=ccrs.PlateCarree(),
                          zorder=3)

# Plot top 100 cities as blue dots
ax.scatter(
    top_1000_ll['lng'],
    top_1000_ll['lat'],
    color='black',
    s=2,
    label='Top 1000 Cities',
    transform=ccrs.PlateCarree(),
    alpha=0.3,
    zorder=4
)

# Plot each trajectory with color mapped to AltP_meters_0
for idx, row in df.iterrows():
    lat_cols = row.filter(regex="Latitude*").values
    lon_cols = row.filter(regex="Longitude*").values

    # Split and plot each segment
    segments = split_trajectory(lat_cols, lon_cols)
    for segment in segments:
        if len(segment) > 1:
            ax.plot(
                segment[:, 0],
                segment[:, 1],
                color=cmap(norm(row['AltP_meters_0'])),  # Map altitude to color
                linewidth=0.5,
                transform=ccrs.PlateCarree(),
                zorder=5
            )


# Plot the flight data on top
sc = ax.scatter(
    df['Longitude_0'].values, 
    df['Latitude_0'].values, 
    c=df['domain_indicator'].values,  # Use numeric values for consistent coloring
    cmap=custom_tab20_21, 
    s=2, 
    transform=ccrs.PlateCarree(),
    zorder=6
)


# Add basemap features
ax.coastlines(resolution='50m')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look


plt.title("Input Space")
# Save and display
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\input_space_w_koppen.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Step 1: Define Reference Points
# A) Average dataset center (mean latitude and longitude)
# center_lat = df['Latitude_0'].mean()
# center_lon = df['Longitude_0'].mean()
# dataset_center = (center_lat, center_lon)

point_90_lonmin = (90, -247.988)

point_neg90_lonmin = (-90, -247.988)

# point_90_neg180 = (90, -143.7789)

# point_neg90_neg180 = (-90, -143.7789)

point_90_lonmax = (90, -14.42)

point_neg90_lonmax = (-90, -14.42)

# List of reference points
reference_points = [point_90_lonmin, point_neg90_lonmin, point_90_lonmax, point_neg90_lonmax]

# Step 2: Define Function to Calculate Distances to Each Reference Point
def calculate_distances_to_references(point):
    """Calculate the distances from a given point to each reference point."""
    distances = [geodesic(point, ref_point).kilometers for ref_point in reference_points]
    return distances

# Step 3: Run Distance Calculations in Parallel for Each Point in `df`
# Prepare points from df as a list of tuples (lat, lon) for each point
points = list(zip(df['Latitude_0'], df['Longitude_0']))

# Parallel computation of distance matrix
distances_extent_corners = Parallel(n_jobs=-1)(
    delayed(calculate_distances_to_references)(point) for point in points
)
dist_matrix_extents = np.array(distances_extent_corners)

# Step 4: Compute Minimum and Mean Distances for Each Point and Add to `df`
df['min_distance_to_extent_corners'] = dist_matrix_extents.min(axis=1)
df['mean_distance_to_extent_corners'] = dist_matrix_extents.mean(axis=1)

In [ ]:
# Step 1: Define Reference Points
# A) Average dataset center (mean latitude and longitude)
# point_90_lonmin = (90, -247.988)

# point_neg90_lonmin = (-90, -247.988)

point_90_neg180 = (90, -180)

point_0_neg180 = (0, -180)

point_neg90_neg180 = (-90, -180)

# List of reference points
reference_points = [point_90_neg180, point_0_neg180, point_neg90_neg180]

# Step 2: Define Function to Calculate Distances to Each Reference Point
def calculate_distances_to_references(point):
    """Calculate the distances from a given point to each reference point."""
    distances = [geodesic(point, ref_point).kilometers for ref_point in reference_points]
    return distances

# Step 3: Run Distance Calculations in Parallel for Each Point in `df`
# Prepare points from df as a list of tuples (lat, lon) for each point
points = list(zip(df['Latitude_0'], df['Longitude_0']))

# Parallel computation of distance matrix
distances_antimeridian = Parallel(n_jobs=-1)(
    delayed(calculate_distances_to_references)(point) for point in points
)
distances_antimeridian = np.array(distances_antimeridian)

# Step 4: Compute Minimum and Mean Distances for Each Point and Add to `df`
df['min_distance_to_antimeridian'] = distances_antimeridian.min(axis=1)
df['mean_distance_to_antimeridian'] = distances_antimeridian.mean(axis=1)

In [ ]:
# Add in new temporal features

# Extract Julian day
df['julian_day'] = df['DateTime_0'].dt.dayofyear

# Determine if the year is a leap year
df['is_leap_year'] = (df['DateTime_0'].dt.year % 4 == 0) & ((df['DateTime_0'].dt.year % 100 != 0) | (df['DateTime_0'].dt.year % 400 == 0))

# Apply cyclic encoding with correct divisor
df['cos_julian_day'] = np.cos(2 * np.pi * df['julian_day'] / df['is_leap_year'].map({True: 366, False: 365}))

In [ ]:
df.to_csv(rf"C:\Users\vwgei\Documents\PVOCAL\data\v{version_str}\df_preprocessed.csv", index=False)

In [ ]:
tt_split = 0.2

# current "ensemble
# Controls the severity of the mwfl parameter for regional ensemble
epsilon = {'CO' : 15, 'CH3Br' : 10, 'O3' : 1, 'DMS' : 6, 'Ethane' : 10, 'CH4' : 1.25}
# Controls the wight of global vs regional ensemble
alpha = {'CO' : 0.40, 'CH3Br' : 0.45, 'O3' : 0.50, 'DMS' : 0.55, 'Ethane' : 0.50, 'CH4' : 0.50}
# Whether or not PASTEL is trained on a log transformed version of the variable (recommended: False)
log_transform = {'CO' : False, 'CH3Br' : False, 'O3' : False, 'DMS' : False, 'Ethane' : False, 'CH4' : False}
# Whether or not to weight samples based on an log transformed scale
sample_weighting = {'CO' : True, 'CH3Br' : True, 'O3' : True, 'DMS' : True, 'Ethane' : True, 'CH4' : True}
# Min weight fraction leaf for the global model
m_w_f_l = {'CO' : 0.0011, 'CH3Br' : 0.0005, 'O3' : 0.0004, 'DMS' : 0.00015, 'Ethane' : 0.0008, 'CH4' : 0.001}

# Aggressive ethane parameters
# epsilon = {'Ethane' : 0.3, }
# alpha = {'Ethane' : 0.50, }
# log_transform = {'Ethane' : False,}
# sample_weighting = {'Ethane' : True,}
# m_w_f_l = {'Ethane' : 0.00005,}

sub_member_size = 25

GMI_flag = True
tree_plotting = False
show_plots_flag = False

# The interval new sample weights will be scaled to
new_min = 1
new_max = 2

# New temporal features for revisions: 'cos_julian_day', 'hour_cos_0', 

# 'min_distance_to_cluster_centroids'
input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation',  'domain_indicator',
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

In [ ]:
ats = ['Ethane'] # 'O3', 'CH4', 'CO', 'Ethane', 'DMS', 'CH3Br'

In [ ]:
r2_scores = []
mse_scores = []

cluster_stats = {}

models_rf = {}
models_lr = {}
best_models = {}  # To store the best models for each region

In [ ]:
PASTEL_train = pd.DataFrame()
PASTEL_test = pd.DataFrame()

In [ ]:
for i in range(len(ats)):
    labelparts = ats[i].split('_')

    Metrics_table_test = pd.DataFrame()
    Metrics_table_train = pd.DataFrame()

    if (len(labelparts) == 3):
        ats_label = labelparts[1]
    else:
        ats_label = labelparts[0]

    global_plot_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "global_models", f"v{version_str}_{ats_label}", "plots")
    global_data_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL","ensemble", "global_models", f"v{version_str}_{ats_label}", "data")
    global_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "global_models", f"v{version_str}_{ats_label}", "models") 
    os.makedirs(global_plot_save_path, exist_ok=True)
    os.makedirs(global_data_save_path, exist_ok=True)
    os.makedirs(global_model_save_path, exist_ok=True)

    print(f"Current Model Run: {ats_label}")

    var_of_int = ats[i]

    # Match unit labels to VOC
    if var_of_int == 'DMS':
        unit_label = "pptv"
        GMI_flag = False
    elif var_of_int == 'CH4':
        unit_label = "ppbv"
        GMI_flag = True
    elif var_of_int == 'CO': 
        unit_label = "ppbv"
        GMI_flag = True
    elif var_of_int == 'O3': 
        unit_label = "ppbv"
        GMI_flag = True
    else:
        unit_label = "pptv"
        if var_of_int == 'Ethane':
            GMI_flag = True
        if var_of_int == 'CH3Br':
            GMI_flag = True

    # Begin Global Model Code
    global_subset = df.dropna(subset=var_of_int)
    global_subset_i = df[~df.index.isin(global_subset.index)]

    global_subset = global_subset.replace([np.inf, -np.inf], np.nan)
    global_subset_i = global_subset_i.replace([np.inf, -np.inf], np.nan)

    global_subset = global_subset.dropna(subset=input_features)
    global_subset_i = global_subset_i.dropna(subset=input_features)

    global_columns = global_subset.columns

    prediction_var = global_subset[var_of_int]
    prediction_var_i = global_subset_i[var_of_int]

    if log_transform[ats_label]:
        prediction_var = np.log1p(prediction_var) # Log Transform
        prediction_var_i = np.log1p(prediction_var_i) # Log Transform

    predictionBaseVars = global_subset[input_features]
    predictionBaseVars_i = global_subset_i[input_features]
    tiny_cluster_65_flag = False
    try:
        print(f"Unique clusters: {len(global_subset['koppen_spatial_relabel_reclass'].unique())}")
        X_train, X_test, y_train, y_test = train_test_split(predictionBaseVars, prediction_var, 
                                                            test_size=tt_split, random_state=None,
                                                            stratify=global_subset['koppen_spatial_relabel_reclass'])
    except ValueError:
        print("[WARNING!] Cannot stratify train test split based on tiny cluster definition of 25")
        print("Using tiny cluster definition of 65")
        print(f"Unique clusters: {len(global_subset['koppen_spatial_relabel_reclass_small'].unique())}")
        X_train, X_test, y_train, y_test = train_test_split(predictionBaseVars, prediction_var, 
                                                            test_size=tt_split, random_state=None,
                                                            stratify=global_subset['koppen_spatial_relabel_reclass_small'])
        tiny_cluster_65_flag = True

    # global_split_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble")
    # os.makedirs(global_split_save_path, exist_ok=True)

    # Save so we can use in the regional model later!
    # Extract the 'iindex' values corresponding to the rows in X_train and X_test
    X_train_iindex = df.loc[X_train.index, 'iindex'].values  # Preserve order
    X_test_iindex = df.loc[X_test.index, 'iindex'].values  # Preserve order

    # Save 'iindex' values separately
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_train_iindex.pkl"), "wb") as f:
        pickle.dump(X_train_iindex, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_test_iindex.pkl"), "wb") as f:
        pickle.dump(X_test_iindex, f)  

    with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_train.pkl"), "wb") as f:
        pickle.dump(X_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_test.pkl"), "wb") as f:
        pickle.dump(X_test, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_y_train.pkl"), "wb") as f:
        pickle.dump(y_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_y_test.pkl"), "wb") as f:
        pickle.dump(y_test, f)

    global_scaler = StandardScaler()
    X_train_scaled = global_scaler.fit_transform(X_train)
    X_test_scaled = global_scaler.transform(X_test)

    train_nd, samp_weights_train, train_kde = calculate_and_inverse_transform_kde(y_train, min_val=new_min, max_val=new_max)

    with open(os.path.join(global_data_save_path, f"{ats_label}_global_samp_weights_train.pkl"), "wb") as f:
        pickle.dump(samp_weights_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_samp_weights_kde_train.pkl"), "wb") as f:
        pickle.dump(train_kde, f)

    # Plot the sample weights
    plt.figure(figsize=(10, 6))
    plt.scatter(train_nd, samp_weights_train, color='cornflowerblue', alpha=0.5, label=ats_label)
    plt.title(f'{ats_label} Sample Weights vs Concentration')
    plt.xlabel(f'{ats_label} log({unit_label})')
    plt.ylabel(f'Sample Weight')
    # plt.legend()
    plt.grid()
    os.makedirs(global_plot_save_path, exist_ok=True)  # Create the directory if it doesn't exist
    plt.savefig(os.path.join(global_plot_save_path, f"sampweight_global.png"), dpi=300, bbox_inches='tight')
    plt.clf()
    plt.close()

    # Global Sample Size
    n = len(global_subset)

    # Initialize Global Model
    PASTEL_Global = RandomForestRegressor(n_estimators=1000,criterion='absolute_error',max_depth=None,
                                        min_samples_split=2,min_samples_leaf=1, max_features=.8,
                                        bootstrap=True, oob_score=True, min_weight_fraction_leaf=m_w_f_l[ats_label],
                                        max_leaf_nodes=None, verbose=0, n_jobs=10)

    LR_Global = LinearRegression(n_jobs=10)

    # Calculate the start time
    global_fit_start_time = time.time()

    print(f"Global model fit in progress...")
    if sample_weighting[ats_label]:
        PASTEL_Global.fit(X_train, y_train.values.ravel(), samp_weights_train)
        LR_Global.fit(X_train, y_train, samp_weights_train)
    else:
        PASTEL_Global.fit(X_train, y_train.values.ravel())
        LR_Global.fit(X_train, y_train)

    # Calculate the end time and time taken
    global_fit_end_time = time.time()
    global_fit_dauration = global_fit_end_time - global_fit_start_time

    # Show the results : this can be altered however you like
    print(f"Fitting the global model took {global_fit_dauration:.2} seconds!")
    print(f"Global fit complete...")

    # Save global model as joblib
    dump(PASTEL_Global, os.path.join(global_model_save_path, f"{ats_label}_Global_RF.joblib"))
    dump(LR_Global, os.path.join(global_model_save_path, f"{ats_label}_Global_LR.joblib"))

    # Predict over Train AND Test
    y_pred_train = PASTEL_Global.predict(X_train)
    y_pred_test = PASTEL_Global.predict(X_test)

    # Predict over full dataset
    y_free_predict = PASTEL_Global.predict(predictionBaseVars)

    # Predict over missing data
    y_inverse_free_predict = PASTEL_Global.predict(predictionBaseVars_i)
    LR_free_predict = LR_Global.predict(predictionBaseVars_i)

    # Save results from predicting over training and testing sets
    np.save(os.path.join(global_data_save_path, f"{ats_label}_global_y_pred_train.npy"), y_pred_train)
    np.save(os.path.join(global_data_save_path, f"{ats_label}_global_y_pred_test.npy"), y_pred_test)

    # Calculate Residuals for use later
    residuals_train = y_train - y_pred_train
    residuals_test = y_test - y_pred_test

    # Save the residuals so we can plot with them later
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_train_residuals.pkl"), "wb") as f:
        pickle.dump(residuals_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_test_residuals.pkl"), "wb") as f:
        pickle.dump(residuals_test, f)

    # Calculate Model Metrics using our function
    PASTEL_Global_train = calculate_and_organize_metrics_global(y_train, y_pred_train, ats_label)
    PASTEL_Global_test = calculate_and_organize_metrics_global(y_test, y_pred_test, ats_label)

    # Apply inverse log transforms if applicable
    if log_transform[ats_label]:
        y_train_i = np.expm1(y_train)
        y_pred_train_i = np.expm1(y_pred_train)
        y_test_i = np.expm1(y_test)
        y_pred_test_i = np.expm1(y_pred_test)
        y_inverse_free_predict_i = np.expm1(y_inverse_free_predict)
        y_free_predict_i = np.expm1(y_free_predict)
        LR_free_predict_i = np.expm1(LR_free_predict)
    else:
        y_train_i = y_train
        y_pred_train_i = y_pred_train
        y_test_i = y_test
        y_pred_test_i = y_pred_test
        y_free_predict_i = y_free_predict
        LR_free_predict_i = LR_free_predict
        y_inverse_free_predict_i = y_inverse_free_predict

    # Visulize predictions on training set
    title = f'Training Results: {ats_label}\n[Global Model]'
    scatter_plot(y_train_i, y_pred_train_i, title, ats_label, unit_label, False, os.path.join(global_plot_save_path, f"{ats_label}_train_scatter.png"))
    # Visulize predictions on testing set
    title = f'Testing Results: {ats_label}\n[Global Model]'
    scatter_plot(y_test_i, y_pred_test_i, title, ats_label, unit_label, False, os.path.join(global_plot_save_path, f"{ats_label}_test_scatter.png"))

    print('Random Forest Out of Bag Score: ')
    print(round(PASTEL_Global.oob_score_,3))

    # Calculate q statistic to get a sense for spatially stratified heterogenaity
    if tiny_cluster_65_flag:
        q_statistic, SSW, SST = calculate_q_statistic(df, df[var_of_int], 'koppen_spatial_relabel_reclass_small')
        print(f"Q-statistic (PBV): {q_statistic}, SSW: {SSW}, SST: {SST}")
    else:
        q_statistic, SSW, SST = calculate_q_statistic(df, df[var_of_int], 'koppen_spatial_relabel_reclass')
        print(f"Q-statistic (PBV): {q_statistic}, SSW: {SSW}, SST: {SST}")

    with open(os.path.join(global_data_save_path, f"{ats_label}_q_stat_train.pkl"), "wb") as f:
        pickle.dump(q_statistic, f)

    # Create a new dataset that just has latitude and longitude information for the training and testing sets
    # - mainly for convience
    latitude_longitude_subset_train = X_train[['Latitude_0', 'Longitude_0']]
    latitude_longitude_subset_test = X_test[['Latitude_0', 'Longitude_0']]

    # Calculate Moran's I
    moran_I_train, p_value_train = calculate_morans_I(latitude_longitude_subset_train, residuals_train, threshold=1)
    moran_I_test, p_value_test = calculate_morans_I(latitude_longitude_subset_test, residuals_test, threshold=1)
    print(f"Moran's I [Train]: {moran_I_train}, p-value: {p_value_train}")
    print(f"Moran's I [Test]: {moran_I_test}, p-value: {p_value_test}")

    with open(os.path.join(global_data_save_path, f"{ats_label}_moran_i_train.pkl"), "wb") as f:
        pickle.dump(moran_I_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_moran_i_p_val_train.pkl"), "wb") as f:
        pickle.dump(p_value_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_moran_i_test.pkl"), "wb") as f:
        pickle.dump(moran_I_test, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_moran_i_p_val_test.pkl"), "wb") as f:
        pickle.dump(p_value_test, f)

        
    # Print out the prediction score of the Linear Regression model that was trained in parallel
    print()
    print(f"Standard Linear Regression Score: {round(LR_Global.score(X_test,y_test), 3)}")

    # Save Moran's I and Moran's I Pvalue and the Linear Regression model accuracy
    PASTEL_train['moran_I_Train'] = moran_I_train
    PASTEL_train['moran_pval_Train'] = p_value_train
    PASTEL_train['LR_Train'] = round(LR_Global.score(X_train, y_train), 3)

    PASTEL_test['moran_I_Test'] = moran_I_test
    PASTEL_test['moran_pval_Test'] = p_value_test
    PASTEL_test['LR_Test'] = round(LR_Global.score(X_test, y_test), 3)

    # Concatenate the result to Metrics_Table
    Metrics_table_train = pd.concat([Metrics_table_train, PASTEL_train], ignore_index=True)
    Metrics_table_test = pd.concat([Metrics_table_test, PASTEL_test], ignore_index=True)

    # Save our metrics table as a pickle object to use later in visualization
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_Metrics_table_train.pkl"), "wb") as f:
        pickle.dump(Metrics_table_train, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_Metrics_table_test.pkl"), "wb") as f:
        pickle.dump(Metrics_table_test, f)

    # Create correlation matrix
    correlation_matrix = predictionBaseVars.corr()

    # Compute total absolute correlation per variable
    total_corr = correlation_matrix.abs().sum(axis=1)  # Sum across rows

    # 3. Sort the variable names by total correlation (least to most)
    sorted_vars = total_corr.sort_values().index

    sorted_corr_matrix = correlation_matrix.loc[sorted_vars, sorted_vars]

    # Plot heatmap with standardized color range and viridis colormap
    plt.figure(figsize=(10, 8))
    sns.heatmap(sorted_corr_matrix, annot=True, cmap='viridis', vmin=-1, vmax=1, linewidths=0.5)

    # Rotate tick labels for better visibility
    plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels (column names)
    plt.yticks(rotation=0)  # Optional: You can rotate y-axis labels if needed

    # Add title and show plot
    plt.title('Sorted Input Variable Correlation (Least to Most Total Pearson Correlation)')
    plt.tight_layout()
    plt.savefig(os.path.join(global_plot_save_path, f"input_vars_correlation_v{version_str}.png"), dpi=300, bbox_inches='tight')
    # if show_plots_flag:
    #     plt.show()
    plt.clf()

    ##############################################################################################################
    # Generate permutation importances using function
    pm_title =f"Permutation Importances using Test: {ats_label}"
    perm_importances_global, perm_importances_index_global = plot_permutation_importance_scores(PASTEL_Global, X_test, y_test, pm_title, os.path.join(global_plot_save_path, f"{ats_label}_global_test_perm_importances.png"))

    # Save importances with their index so we can create a cool plot later!
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_perm_importances.pkl"), "wb") as f:
        pickle.dump(perm_importances_global, f)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_perm_importances_index.pkl"), "wb") as f:
        pickle.dump(perm_importances_index_global, f)

    # Get genaric (gini impurity) feature importances 
    feature_importances = PASTEL_Global.feature_importances_

    # Get indices of the two most important features
    top_two_indices = np.argsort(perm_importances_global['importances_mean'])[-2:] # IF WE GET AN ERROR HERE CHANGE BACK TO OLD IMPORTANCES
    top_two_permutation_indeces = perm_importances_index_global[-2:]
    # print()

    # Print the indices and names of the two most important features
    print("Feature names:", X_test.columns[top_two_permutation_indeces])

    #From SKlearn PDP tutorial!   
    top_two_feat = X_test.columns[top_two_permutation_indeces] 
        
    features_info_top_two = {
        "features": [top_two_feat[0], top_two_feat[1],(top_two_feat[0], top_two_feat[1])],
        "kind" : "average",
    }
    _, ax = plt.subplots(ncols=3, figsize=(10, 4), constrained_layout=True)
                
    # common_params = {
    #     "sample_weight":samp_weights_train
    #     }

    display = PartialDependenceDisplay.from_estimator(
    PASTEL_Global,
    X_test,
    **features_info_top_two,
    ax=ax,
    # **common_params,
    )
            
    _ = display.figure_.suptitle(f"1-way vs 2-way Partial Dependance\n [{ats_label}] {top_two_feat[0]} and {top_two_feat[1]})", fontsize=16) #str(i+-24)
    plt.savefig(os.path.join(global_plot_save_path, f"{ats_label}_global_pdp.png"), dpi=300, bbox_inches='tight') #str(i+-24)
    plt.show()



    # Check to see if this works and if we can replace the old code above.
    # mapcorners = [-260, -90, 20, 90]  # Full globe

    # Adjust longitudes for plotting
    ilongs = global_subset_i["Longitude_0"].values
    plongs = global_subset["Longitude_0"].values
    ilats = global_subset_i["Latitude_0"].values
    plats = global_subset["Latitude_0"].values

    ilongs[ilongs > 100] -= 360
    plongs[plongs > 100] -= 360

    # Create the figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8), 
                                subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})

    # Define a function to format each subplot
    def setup_map(ax, title):
        ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))
        ax.coastlines(resolution='50m')
        ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
        ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')
        
        # Add gridlines
        gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
        gl.top_labels = gl.right_labels = False  # Hide top/right labels
        ax.set_title(title)

    # Setup both subplots
    setup_map(ax1, "Measured Data")
    setup_map(ax2, "PASTEL (global) Modeled Missing Data")

    # Determine colormap normalization
    if var_of_int in ['Ethane', 'O3', 'DMS', 'CO']:
        norm = plt.Normalize(vmin=max(1e-3, min(y_free_predict_i.min(), y_inverse_free_predict_i.min())), 
                            vmax=max(y_free_predict_i.max(), y_inverse_free_predict_i.max()))
    else:
        norm = plt.Normalize(vmin=min(y_free_predict_i.min(), y_inverse_free_predict_i.min()), 
                            vmax=max(y_free_predict_i.max(), y_inverse_free_predict_i.max()))

    # Plot the True ATom Data (first subplot)
    sc1 = ax1.scatter(plongs, plats, s=20, c=y_free_predict_i, cmap='plasma', norm=norm, 
                    edgecolor='none', transform=ccrs.PlateCarree())

    # Plot the PASTEL Modeled Missing Data (second subplot)
    sc2 = ax2.scatter(ilongs, ilats, s=20, c=y_inverse_free_predict_i, cmap='plasma', norm=norm, 
                    edgecolor='none', transform=ccrs.PlateCarree())

    # Add a shared colorbar
    cbar = fig.colorbar(sc1, ax=[ax1, ax2], orientation='horizontal', fraction=0.04, pad=0.1)
    cbar.set_label(f'Concentration of {var_of_int} {unit_label}')

    # Save and optionally display
    plt.savefig(os.path.join(global_plot_save_path, f"{ats_label}_Global_Modeled_Missing_Data_v{version_str}.png"), dpi=300, bbox_inches='tight')
    if show_plots_flag:
        plt.show()

    plt.clf()
    plt.close()

    # Make sure our lists are empty when starting a new ats for regional model
    n_samples_list = []

    clusters_skipped = 0
    skipped_clusters = []

    if tiny_cluster_65_flag:
        relabel_reclass_str = 'koppen_spatial_relabel_reclass_small'
    else:
        relabel_reclass_str = 'koppen_spatial_relabel_reclass'

    for cluster in np.sort(df[relabel_reclass_str].unique()):
        # Generate paths for regional ensemble code
        cluster_plot_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "regional_ensemble", f"v{version_str}_{ats_label}", "plots", f"{cluster}")
        cluster_data_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL","ensemble", "regional_ensemble", f"v{version_str}_{ats_label}", "data", f"{cluster}")
        cluster_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "regional_ensemble", f"v{version_str}_{ats_label}", "models", f"{cluster}")
        os.makedirs(cluster_plot_save_path, exist_ok=True)
        os.makedirs(cluster_data_save_path, exist_ok=True)
        os.makedirs(cluster_model_save_path, exist_ok=True)

        # Open the file in binary read mode and load the object
        with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_train.pkl"), 'rb') as file:
            temp_X_train = pickle.load(file)
        with open(os.path.join(global_data_save_path, f"{ats_label}_global_y_train.pkl"), 'rb') as file:
            temp_y_train = pickle.load(file)
        with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_train_iindex.pkl"), 'rb') as file:
            temp_X_train_iindex = pickle.load(file)

        temp_df = pd.concat([temp_X_train, temp_y_train], axis=1)

        temp_df['iindex'] = temp_X_train_iindex

        # Ensure the indexes of combined_df are used to filter df
        spatial_cluster_subset = temp_df.merge(df[['iindex', relabel_reclass_str]], on='iindex', how='left')

        # At this point our temporary dataset has both our "var_of_int", "iindex", and our koppen classificaiton
        #  all of which we will need to remove before training the regional model.

        # # Add this column to combined_df if needed
        # temp_df['koppen_spatial_relabel_reclass'] = spatial_cluster_subset

        cluster_df = spatial_cluster_subset[spatial_cluster_subset[relabel_reclass_str] == cluster]
        print(f"Processing ensemble member: {cluster}")

        # Drop any rows with nans in the var_of_int
        data_subset = cluster_df.dropna(subset=var_of_int)
        # inverse_subset = cluster_df[~cluster_df.index.isin(data_subset.index)]

        # Just in case?
        data_subset = data_subset.replace([np.inf, -np.inf], np.nan)
        # inverse_subset = inverse_subset.replace([np.inf, -np.inf], np.nan)

        # Drop any rows with nans in the input_features
        data_subset = data_subset.dropna(subset=input_features)
        # inverse_subset = inverse_subset.dropna(subset=input_features)

        cluster_iindex = data_subset['iindex']
        with open(os.path.join(cluster_data_save_path, "cluster_iindex_train.pkl"), "wb") as f:
            pickle.dump(cluster_iindex, f)

        # We don't need to drop them because we just grab the columns we need anyway?
        # data_subset.drop(columns=['iindex', var_of_int, 'koppen_spatial_relabel_reclass'], inplace=True)

        # columns = data_subset.columns

        n = len(data_subset)
        if n < 3:
            print(f"Less than 3 points in cluster: {cluster}. Unable to generate sample weights. Skipping cluster...")
            clusters_skipped += 1
            skipped_clusters.append(cluster)
            continue  

        if n < 100:
            print(f"Cluster too small, performing bootstrapping up to 100 samples...")

            # Bootstrapping: Sample with replacement
            bootstrapped_sample = data_subset.sample(n=100, replace=True, random_state=42)
            print(f"Original data size: {len(data_subset)}")
            print(f"Bootstrapped sample size: {len(bootstrapped_sample)}")

            # Apply Gaussian noise to numerical features of all bootstrapped samples
            numerical_features = [
                'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
                'Latitude_0', 'Longitude_0', 'AltP_meters_0',
                'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
                'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation',
                'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
            ]

            # Calculate feature-wise standard deviation for noise scaling
            feature_std = bootstrapped_sample[numerical_features].std()
            # print("Feature-wise standard deviation for Gaussian noise:")
            # print(feature_std)

            if ats_label == "DMS":
                noise = np.random.normal(0, 0.025 * feature_std, bootstrapped_sample[numerical_features].shape)
            else:
                # Generate Gaussian noise for numerical features
                noise = np.random.normal(0, 0.05 * feature_std, bootstrapped_sample[numerical_features].shape)

            # Add noise to numerical features
            bootstrapped_sample[numerical_features] += noise

            # Update data_subset to the augmented bootstrapped sample
            data_subset = bootstrapped_sample

            # Now data_subset has 100 samples with Gaussian noise added
            num_bootstrapped = len(data_subset) - n
            print(f"Number of samples bootstrapped and augmented with Gaussian noise: {num_bootstrapped}")
            n_samples_list.append(len(data_subset))
        else:
            num_bootstrapped = 0

        total_cluster_samples = n + num_bootstrapped

        print(f"Cluster sample size: {total_cluster_samples}")
        n_samples_list.append(total_cluster_samples)

        # Target var and perform log transform
        prediction_var = data_subset[var_of_int]
        if log_transform[ats_label]:
            prediction_var = np.log1p(prediction_var) # Log Transform
        
        predictionBaseVars = data_subset[input_features]

        with open(os.path.join(cluster_data_save_path, "predictionBaseVars_wgn.pkl"), "wb") as f:
            pickle.dump(predictionBaseVars, f)

        # intialize empty list for storing submember information
        test_r2_temp = []
        train_r2_temp =[]
        test_r2_temp_LR = []
        train_r2_temp_LR = []

        lr_sm_models = []
        rf_sm_models = []

        # Loop through submembers
        for j in range(sub_member_size):
            sub_member = j
            # Initalize paths for saving submember outputs
            sub_member_data_path = os.path.join(cluster_data_save_path, f"{sub_member}")
            os.makedirs(sub_member_data_path, exist_ok=True)
            sub_member_model_path = os.path.join(cluster_model_save_path, f"{sub_member}")
            os.makedirs(sub_member_model_path, exist_ok=True)

            # Perform train/test split on the submember
            X_train, X_test, y_train, y_test = train_test_split(predictionBaseVars, prediction_var, 
                                                                test_size=tt_split, random_state=None)
            
            # xtrain_gdas_temp = X_train[X_train['domain_indicator'] == 1]
            # xtrain_era5_temp = X_train[X_train['domain_indicator'] == 0]

            # Standardize input variables to between -1 and 1
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Calculate submember sample weights
            train_nd, samp_weights_train, train_kde = calculate_and_inverse_transform_kde(y_train, min_val=new_min, max_val=new_max)

            # Save all submember information
            np.save(os.path.join(sub_member_data_path, f"sample_weights_train_sm{sub_member}.npy"), samp_weights_train)
            np.save(os.path.join(sub_member_data_path, f"X_train_temp_sm{sub_member}.npy"), X_train)
            np.save(os.path.join(sub_member_data_path, f"X_test_temp_sm{sub_member}.npy"), X_test)
            np.save(os.path.join(sub_member_data_path, f"y_train_temp_sm{sub_member}.npy"), y_train)
            np.save(os.path.join(sub_member_data_path, f"y_test_temp_sm{sub_member}.npy"), y_test)

            # # Plot the sample weights
            # plt.figure(figsize=(10, 6))
            # plt.scatter(train_nd, samp_weights_train, color='cornflowerblue', alpha=0.5, label=ats_label)
            # plt.title(f'{ats_label} Sample Weights vs Concentration')
            # plt.xlabel(f'{ats_label} {ats_label}')
            # plt.ylabel(f'Sample Weight')
            # # plt.legend()
            # plt.grid()
            # os.makedirs(sub_member_plot_path, exist_ok=True)  # Create the directory if it doesn't exist
            # plt.savefig(os.path.join(sub_member_plot_path, f"sampweight_{iteration}.png"))
            # plt.clf()
            
            # Use epsilon to tune our Min_weight_fraction_leaf parameter to account for submember size
            mwfl = epsilon[ats_label] / n
            if mwfl > 0.5:
                print("[WARNING] Too few samples - MWFL has been adjusted")
                mwfl = 0.5

            # Initialize the submember model
            random_number = random.randint(1, 1000000)
            sm_PASTEL_EST = RandomForestRegressor(n_estimators=100,criterion='absolute_error',max_depth=None,
                                        min_samples_split=2,min_samples_leaf=1, max_features=3,
                                        min_weight_fraction_leaf=mwfl,verbose=0,n_jobs=10, random_state=random_number
                                        # bootstrap=True, oob_score=True
                                        )
            # Also Initialize a linear model for the submember
            sm_LR = LinearRegression(n_jobs=10)
            print(f"\tTraining ensamble sub-member: [{sub_member}]")

            # Checks to see if sample weighting should be performed for this ATS
            if sample_weighting[ats_label]:
                sm_PASTEL_EST.fit(X_train, y_train.values.ravel(), samp_weights_train)
                sm_LR.fit(X_train, y_train, samp_weights_train)
            else:
                sm_PASTEL_EST.fit(X_train, y_train.values.ravel())
                sm_LR.fit(X_train, y_train)
            
            # Append the sub member model to the list of models
            lr_sm_models.append(sm_LR)
            rf_sm_models.append(sm_PASTEL_EST)

            # Save the model just in case it is needed later
            dump(sm_PASTEL_EST, os.path.join(sub_member_model_path, f"c_{cluster}_sm_{sub_member}_RF.joblib"))
            dump(sm_LR, os.path.join(sub_member_model_path, f"c_{cluster}_sm_{sub_member}_LR.joblib"))

        # Initialize arrays to store predictions
        predictions_train_RF = np.zeros((sub_member_size, X_train.shape[0]))
        predictions_test_RF = np.zeros((sub_member_size, X_test.shape[0]))
        predictions_train_LR = np.zeros((sub_member_size, X_train.shape[0]))
        predictions_test_LR = np.zeros((sub_member_size, X_test.shape[0]))

        # Store submember precition information within array
        for k, sm_model in enumerate(rf_sm_models):
            predictions_train_RF[k] = sm_model.predict(X_train)
            predictions_test_RF[k] = sm_model.predict(X_test)
        for k, sm_model in enumerate(lr_sm_models):
            predictions_train_LR[k] = sm_model.predict(X_train)
            predictions_test_LR[k] = sm_model.predict(X_test)        

        # Average Predictions together for Random Forest 
        train_avg_RF = predictions_train_RF.mean(axis=0)
        test_avg_RF = predictions_test_RF.mean(axis=0)
        # Average Predictions together for Linear Regression
        train_avg_LR = predictions_train_LR.mean(axis=0)
        test_avg_LR = predictions_test_LR.mean(axis=0)

        # Calculate Regional Residuals for Random Forest
        regional_residuals_train_RF = y_train - train_avg_RF
        regional_residuals_test_RF = y_test - test_avg_RF
        # Calculate Regional Residuals for Linear Regression
        regional_residuals_train_LR = y_train - train_avg_LR
        regional_residuals_test_LR = y_test - test_avg_LR


        # Calculate and organize metrics for training and testing set for cluster
        train_metrics_RF = calculate_and_organize_metrics_cluster(y_train, train_avg_RF, cluster, ats_label, train_test="Train")
        test_metrics_RF = calculate_and_organize_metrics_cluster(y_test, test_avg_RF, cluster, ats_label, train_test="Test")
        train_metrics_LR = calculate_and_organize_metrics_cluster(y_train, train_avg_LR, cluster, ats_label, train_test="Train")
        test_metrics_LR = calculate_and_organize_metrics_cluster(y_test, test_avg_LR, cluster, ats_label, train_test="Test")

        # Get the latitude and longitude information of just the subset used for training/testing
        latitude_longitude_subset_train = X_train[['Latitude_0','Longitude_0']] #df[['Latitude_0', 'Longitude_0']].loc[X_train.index]
        latitude_longitude_subset_test = X_test[['Latitude_0','Longitude_0']]#df[['Latitude_0', 'Longitude_0']].loc[X_test.index]

        # Use the above lat lon subsets to calcuate Moran's I
        moran_I_train_RF, p_value_train_RF = calculate_morans_I(latitude_longitude_subset_train, regional_residuals_train_RF, threshold=1)
        moran_I_test_RF, p_value_test_RF = calculate_morans_I(latitude_longitude_subset_test, regional_residuals_test_RF, threshold=1)
        moran_I_train_LR, p_value_train_LR = calculate_morans_I(latitude_longitude_subset_train, regional_residuals_train_LR, threshold=1)
        moran_I_test_LR, p_value_test_LR = calculate_morans_I(latitude_longitude_subset_test, regional_residuals_test_LR, threshold=1)

        # Print out information to the console
        print(f"Moran's I [Train - RF]: {moran_I_train_RF}, p-value: {p_value_train_RF}")
        print(f"Moran's I [Test - RF]: {moran_I_test_RF}, p-value: {p_value_test_RF}")
        print(f"Moran's I [Train - LR]: {moran_I_train_LR}, p-value: {p_value_train_LR}")
        print(f"Moran's I [Test - LR]: {moran_I_test_LR}, p-value: {p_value_test_LR}")
        print()

        # Check which model (RF or LR) had the highest R^2 and designate it as the best for that region
        if test_metrics_RF['R2_Test'].iloc[0] > test_metrics_LR['R2_Test'].iloc[0]:
            best_model = 'RF'
        else:
            best_model = 'LR'

        # Apply inverse log transforms
        if log_transform[ats_label]:
            y_train_i = np.expm1(y_train)
            y_test_i = np.expm1(y_test)
            train_avg_RF_i = np.expm1(train_avg_RF)
            test_avg_RF_i = np.expm1(test_avg_RF)
            train_avg_LR_i = np.expm1(train_avg_LR)
            test_avg_LR_i = np.expm1(test_avg_LR)
        else:
            y_train_i = y_train
            y_test_i = y_test
            train_avg_RF_i = train_avg_RF
            test_avg_RF_i = test_avg_RF
            train_avg_LR_i = train_avg_LR
            test_avg_LR_i = test_avg_LR

        # Visualize predictions on training and testing sets for Random Forest
        scatter_save_path_train = os.path.join(cluster_plot_save_path, f"RF_scatter_train.png")
        title = f'Training Results [{ats_label} Cluster: #{cluster} - RF]'
        scatter_plot(y_train_i, train_avg_RF_i, title, ats_label, unit_label, False, scatter_save_path_train)
        scatter_save_path_test = os.path.join(cluster_plot_save_path, f"RF_scatter_test.png")
        title = f'Testing Results [{ats_label} Cluster: #{cluster} - RF]'
        scatter_plot(y_test_i, test_avg_RF_i, title, ats_label, unit_label, False, scatter_save_path_test)
        # Visualize predictions on training and testing sets for Linear Regression
        scatter_save_path_train = os.path.join(cluster_plot_save_path, f"LR_scatter_train.png")
        title = f'Training Results [{ats_label} Cluster: #{cluster} - LR]'
        scatter_plot(y_train_i, train_avg_LR_i, title, ats_label, unit_label, False, scatter_save_path_train)
        scatter_save_path_test = os.path.join(cluster_plot_save_path, f"LR_scatter_test.png")
        title = f'Testing Results [{ats_label} Cluster: #{cluster} - LR]'
        scatter_plot(y_test_i, test_avg_LR_i, title, ats_label, unit_label, False, scatter_save_path_test)

        # Plot the average permutation feature importance for rf model (style 1) (also more useful)
        mean_importances, std_importances, sorted_idx = plot_average_permutation_importance_scores_1(
            models=rf_sm_models,
            input_x=X_test,
            input_y=y_test,  
            title=f"Permutation Importance Across Cluster Ensemble | {ats_label}",
            save_path=os.path.join(cluster_plot_save_path, f"cluster_{cluster}_permutation_importances_dotline.png")
        )

        # # Plot the average permutation feature importance for rf model (style 2) (may not be working correctly?)
        # mean_importances,  sorted_idx = plot_average_permutation_importance_scores_2(
        #     models=rf_sm_models,
        #     input_x=X_test,  
        #     input_y=y_test,  
        #     title=f"Permutation Importance Across Cluster Ensemble | {ats_label}",
        #     save_path=os.path.join(cluster_plot_save_path, f"cluster_{cluster}_permutation_importances_boxplot.png")
        # )

        # Get the top two most important features
        top_two_feat = sorted_idx[-2:]

        try:
            # Use the top two most important features to plot a 2 way partial dependenace plot
            plot_ensemble_partial_dependence(
                models=rf_sm_models,
                X=X_test, 
                top_features=top_two_feat,
                save_path=os.path.join(cluster_plot_save_path, f"cluster_{cluster}_PDP.png"),
                title=f"Cluster {cluster} {ats_label} Ensemble Partial Dependence"
            )
        except ValueError:
            print(f"[WARNING] - PDP plot failed to generate for cluster {cluster}.")
            print("This will happen when RF predictions are a single constant number accross all samples.")

        # Save regional ensemble/submember models
        temp_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "regional_ensemble", f"v{version_str}_{ats_label}", "models", f"{cluster}")
        all_sm_models_rf = glob.glob(f"{temp_model_save_path}/**/*RF.joblib")
        all_sm_models_lr = glob.glob(f"{temp_model_save_path}/**/*LR.joblib")

        # current_clust_temp_d -= 1
        # current_clust_temp_u += 1
        # if current_clust_temp_d == -1:
        #     current_clust_temp_d = kmeans_k
        # if current_clust_temp_u > kmeans_k:
        #     current_clust_temp_u = 0

        # temp_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "regional_ensemble", f"V4_{ats_label}", "models", f"{current_clust_temp_u}")
        # temp_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "bestmodels", f"V4_{ats_label}", f"{current_clust_temp_d}")
        # all_sm_neighbor_models_rf = glob.glob(f"{temp_model_save_path}/**/*RF.joblib")
        # all_sm_neighbor_models_lr = glob.glob(f"{temp_model_save_path}/**/*LR.joblib")

        # current_clust_temp_d -= 1
        # current_clust_temp_u += 1
        # if current_clust_temp_d == -1:
        #     current_clust_temp_d = kmeans_k
        # if current_clust_temp_u > kmeans_k:
        #     current_clust_temp_u = 0

        # temp_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "bestmodels", f"V4_{ats_label}", f"{current_clust_temp_u}")
        # temp_model_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "bestmodels", f"V4_{ats_label}", f"{current_clust_temp_d}")
        # all_sm_neighbor_neighbor_models_rf = glob.glob(f"{temp_model_save_path}/**/*RF.joblib")
        # all_sm_neighbor_neighbor_models_lr = glob.glob(f"{temp_model_save_path}/**/*LR.joblib")

        # Create a dictionary with everything for easy access later!
        cluster_stats[cluster] = {
            'CLUSTER_ID' : cluster,
            'best_model' : best_model,
            'epsilon' : epsilon[ats_label],
            'alpha' : alpha[ats_label],
            'log_transform' : log_transform[ats_label],
            'sample_weighting' : sample_weighting[ats_label],
            'm_w_f_l' : m_w_f_l[ats_label],
            'n' : n,
            'clusters_skipped' : clusters_skipped,
            'skipped_clusters' : skipped_clusters,
            'num_bootstrapped' : num_bootstrapped,
            'train_metrics_RF' : train_metrics_RF,
            'test_metrics_RF' : test_metrics_RF,
            'train_metrics_LR' : train_metrics_LR,
            'test_metrics_LR' : test_metrics_LR,
            'train_avg_RF' : train_avg_RF,
            'test_avg_RF' : test_avg_RF,
            'train_avg_LR' : train_avg_LR,
            'test_avg_LR' : train_avg_LR,
            'regional_residuals_train_RF' : regional_residuals_train_RF,
            'regional_residuals_test_RF' : regional_residuals_test_RF,
            'regional_residuals_train_LR' : regional_residuals_train_LR,
            'regional_residuals_test_LR' : regional_residuals_test_LR,
            'moran_I_train_RF' : moran_I_train_RF,
            'moran_I_test_RF' : moran_I_test_RF,
            'p_value_train_RF' : p_value_train_RF,
            'p_value_test_RF' : p_value_test_RF,
            'moran_I_train_LR' : moran_I_train_LR,
            'moran_I_test_LR' : moran_I_test_LR,
            'p_value_train_LR' : p_value_train_LR,
            'p_value_test_LR' : p_value_test_LR,
            'mean_importances' : mean_importances,
            'sorted_idx' : sorted_idx,
            'top_two_feat' : top_two_feat,
            'all_sm_models_RF' : all_sm_models_rf,
            'all_sm_models_LR': all_sm_models_lr,
            'q_statistic' : q_statistic,
            'SST' : SST,
            'SSW' : SSW,
        }

        # Print out information for RF and LR models for a particular cluster
        print(f"Cluster Ensamble Score [Random Forest]: {np.round(test_metrics_RF['R2_Test'].iloc[0],3)}")
        print(f"Cluster Ensamble Score [Linear Regression]: {np.round(test_metrics_LR['R2_Test'].iloc[0],3)}")
        print()
        print(f"Best Model for Cluster {cluster}: {best_model}")
        print("-----------------------------------------------------")
        
        
    # Save with pickle
    with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats_label}_cluster_stats.pkl", "wb") as f:
        pickle.dump(cluster_stats, f)

    # could be good for a compairison when looking at regions where RF beats LR that region probalby has
    # more data and therefore the model is more complex?

    # This ^^ but actually....


    # if tree_plotting:
    #     # Visualize example decision tree - Increases run time significantly
    #     fig = plt.figure(figsize=(30, 20))
    #     plot_tree(PASTEL_EST.estimators_[0], 
    #                 filled=True, impurity=True, 
    #                 rounded=True)
    #     fig.savefig(os.path.join(plot_save_path, "rfrtree.png"))

    # Loop through each cluster in cluster_stats
    for cluster_id, stats in cluster_stats.items():
        # Check which model was chosen as best for this cluster
        if stats['best_model'] == 'RF':
            r2_scores.append(stats['test_metrics_RF']['R2_Test'].iloc[0])
            mse_scores.append(stats['test_metrics_RF']['MSE_Test'].iloc[0])
        else:  # 'LR' was the best model
            r2_scores.append(stats['test_metrics_LR']['R2_Test'].iloc[0])
            mse_scores.append(stats['test_metrics_LR']['MSE_Test'].iloc[0])

    # Get the average accuracy across all clusters
    average_r2 = np.mean(r2_scores)
    average_mse = np.mean(mse_scores)
    
    print(f'{ats_label} mean sample size is: {np.mean(n_samples_list)}')
    print(f'{ats_label} min sample size is: {np.min(n_samples_list)}')
    print("***************************")
    print(f"{ats_label} Average Ensemble R²: {average_r2:.3f}")
    print(f"{ats_label} Average Ensemble MSE: {average_mse:.3f}")
    print("**************************")
    print("########## Start Regional Ensemble Predictions ##########")
    # Specify the path where the models are stored
    directory = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "regional_ensemble", f"v{version_str}_{ats_label}", "models")

    # Dictionary to store best models by ATS type and meteorological cluster
    models = {}

    # Iterate over cluster_stats to identify and load the best model for each cluster
    for cluster_id, cluster_data in cluster_stats.items():
        # Determine the best model type for this cluster
        best_model_type = cluster_data['best_model']  # e.g., "RF" or "LR"
        model_paths = cluster_data[f'all_sm_models_{best_model_type}']  # Get list of model paths

        # Load each model in the best model path list
        loaded_models = [load(model_path) for model_path in model_paths]
        
        # Initialize the structure in `models` if necessary
        if best_model_type not in models:
            models[best_model_type] = {}
        # Store the loaded models for the spatial cluster
        models[best_model_type][cluster_id] = loaded_models

    # Function to predict using ensemble members/submembers
    def predict_with_ensemble(cluster_stats, ats_type, sample_index, sample_features):
        # Identify the spatial_cluster of the sample
        spatial_cluster = int(df.loc[sample_index, relabel_reclass_str])
        samp_lat = df.loc[sample_index, 'Latitude_0']
        samp_lon = df.loc[sample_index, 'Longitude_0']

        # Get the best model type for this cluster from cluster_stats
        try:
            # This line may fail if the cluster isn't present
            best_model_type = cluster_stats[spatial_cluster]['best_model']

            # Retrieve loaded models for the cluster in the models dictionary
            sub_members = models.get(best_model_type, {}).get(spatial_cluster, [])
            
            # Perform predictions using each model in sub_members
            predictions = [model.predict(sample_features.reshape(1, -1))[0] for model in sub_members]
            
            # Average predictions to get final result
            avg_pred = np.mean(predictions) if predictions else None

        except KeyError:
            print(f"Cluster: [{spatial_cluster}] has been excluded due to low/no data.")
            print("Finding next closest valid cluster to predict from...")

            # Get the original mapping and centroid
            original_mapping = get_key_by_value(cluster_mapping, spatial_cluster)
            original_centroid = centroids_reclass[original_mapping]
            print(f"Original centroid of native cluster: {original_centroid}")

            # Compute distances to all centroids
            distances = {}
            for cluster_id, centroid in centroids_reclass.items():
                if cluster_id != original_mapping:  # Ignore the current cluster
                    dist = haversine_calc(samp_lat, samp_lon, centroid[0], centroid[1])
                    distances[cluster_id] = dist

            # Sort by distance
            sorted_clusters = sorted(distances.items(), key=lambda x: x[1])

            # Find the next closest cluster that exists in cluster_stats
            next_closest_cluster = None
            for cluster_id, _ in sorted_clusters:
                if cluster_id in cluster_mapping and cluster_mapping[cluster_id] in cluster_stats:
                    next_closest_cluster = cluster_mapping[cluster_id]
                    break
            
            # If no valid cluster is found, return None
            if next_closest_cluster is None:
                print("No valid cluster found in cluster_stats.")
                return None
            
            temp_clust = get_key_by_value(cluster_mapping, next_closest_cluster)
            print(f"Next closest valid cluster: {next_closest_cluster} at {centroids_reclass[temp_clust]}")

            # Get the best model type for the next closest cluster
            best_model_type = cluster_stats[next_closest_cluster]['best_model']

            # Retrieve loaded models for the cluster in the models dictionary
            sub_members = models.get(best_model_type, {}).get(next_closest_cluster, [])
            
            # Perform predictions using each model in sub_members
            predictions = [model.predict(sample_features.reshape(1, -1))[0] for model in sub_members]
            
            # Average predictions to get final result
            avg_pred = np.mean(predictions) if predictions else None

        return avg_pred

    # Open the pickled global training and testing sets
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_test.pkl"), 'rb') as file:
        X_test = pickle.load(file)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_y_test.pkl"), 'rb') as file:
        y_test = pickle.load(file)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_train.pkl"), 'rb') as file:
        X_train = pickle.load(file)
    with open(os.path.join(global_data_save_path, f"{ats_label}_global_y_train.pkl"), 'rb') as file:
        y_train = pickle.load(file)

    # Find skipped clusters (due to lack of data) if there are any...
    keys = set(list(cluster_stats.keys()))
    cluster_keys = set([i for i in range(65)])
    missing_keys = list(cluster_keys - keys)
    
    # Lists to store predictions
    test_predictions = []
    train_predictions = []

    # Loop through each test sample and predict using the best model for the sample's cluster
    # for idx, row in tqdm(X_test.iterrows(), total=X_test.shape[0], desc="Predicting on test set"):
    #     # Call the prediction function with appropriate arguments
    #     prediction = predict_with_ensemble(cluster_stats, var_of_int, idx, row[input_features].values)
    #     test_predictions.append(prediction)

    # # Loop through each test sample and predict using the best model for the sample's cluster
    # for idx, row in tqdm(X_train.iterrows(), total=X_train.shape[0], desc="Predicting on train set"):
    #     # Call the prediction function with appropriate arguments
    #     prediction = predict_with_ensemble(cluster_stats, var_of_int, idx, row[input_features].values)
    #     train_predictions.append(prediction)

    mi = 0
    for idx, row in X_test.iterrows():
        # Print progress every 100 steps
        if mi % 100 == 0:
            print(f"Predicting on test set: {mi}/{X_test.shape[0]}")

        # Call the prediction function with appropriate arguments
        prediction = predict_with_ensemble(cluster_stats, var_of_int, idx, row[input_features].values)
        test_predictions.append(prediction)
        mi += 1

    mi = 0
    # Loop through each train sample and predict using the best model for the sample's cluster
    for idx, row in X_train.iterrows():
        # Print progress every 100 steps (you can adjust the frequency as needed)
        if mi % 100 == 0:
            print(f"Predicting on train set: {mi}/{X_train.shape[0]}")

        # Call the prediction function with appropriate arguments
        prediction = predict_with_ensemble(cluster_stats, var_of_int, idx, row[input_features].values)
        train_predictions.append(prediction)
        mi += 1

    # Convert predictions to a numpy array for further processing
    np_test_predictions = np.array(test_predictions)
    np_train_predictions = np.array(train_predictions)

    # Save the test predictions array
    regional_ensemble_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "regional_ensemble", f"v{version_str}_{ats_label}")
    os.makedirs(regional_ensemble_save_path, exist_ok=True)

    # Save information about missing clusters if there are any
    with open(os.path.join(regional_ensemble_save_path, f"{ats_label}_missing_keys.pkl"), "wb") as f:
        pickle.dump(missing_keys, f)

    # Save the regional ensemble predictions!
    np.save(os.path.join(regional_ensemble_save_path, f"{ats_label}_re_test_predictions_native.npy"), np_test_predictions)
    np.save(os.path.join(regional_ensemble_save_path, f"{ats_label}_re_train_predictions_native.npy"), np_train_predictions)

    # Calculate residuals and save them
    re_residuals_test = y_test - test_predictions
    with open(os.path.join(regional_ensemble_save_path, f"{ats_label}_re_residuals_test.pkl"), "wb") as f:
        pickle.dump(re_residuals_test, f)
    # Calculate residuals and save them
    re_residuals_train = y_train - train_predictions
    with open(os.path.join(regional_ensemble_save_path, f"{ats_label}_re_residuals_train.pkl"), "wb") as f:
        pickle.dump(re_residuals_test, f)

    # Inverse transform test and prediction data for scatter plot
    if log_transform[ats_label]:
        y_test_i = np.expm1(y_test)
        test_predictions_i = np.expm1(test_predictions)
        y_train_i = np.expm1(y_train)
        train_predictions_i = np.expm1(train_predictions)
    else:
        y_test_i = y_test
        test_predictions_i = test_predictions
        y_train_i = y_train
        train_predictions_i = train_predictions

    # Visualize predictions on testing set [regional model on global data using native clusters]
    title = f'Testing Results: {ats_label}\n[Regional Ensemble]'
    scatter_plot(y_test_i, test_predictions_i, title, ats_label, unit_label, False, 
                os.path.join(regional_ensemble_save_path, f"Regional_ensemble_{ats_label}_native_test_scatter.png"))

    # Visualize predictions on testing set [regional model on global data using native clusters]
    title = f'Training Results: {ats_label}\n[Regional Ensemble]'
    scatter_plot(y_train_i, train_predictions_i, title, ats_label, unit_label, False, 
                os.path.join(regional_ensemble_save_path, f"Regional_ensemble_{ats_label}_native_train_scatter.png"))
    

    # Combine Global and Local models using local weight (alpha)!
    combined_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL", "ensemble", "PASTEL_combined", f"v{version_str}_{ats_label}")
    os.makedirs(combined_save_path, exist_ok=True)

    # Load in saved information about the global model
    global_pred_train = np.load(os.path.join(global_data_save_path, f"{ats_label}_global_y_pred_train.npy"))
    global_pred_test = np.load(os.path.join(global_data_save_path, f"{ats_label}_global_y_pred_test.npy"))
    local_pred_train = np.load(os.path.join(regional_ensemble_save_path, f"{ats_label}_re_train_predictions_native.npy"))
    local_pred_test = np.load(os.path.join(regional_ensemble_save_path, f"{ats_label}_re_test_predictions_native.npy"))

    # Combine global and regional ensemble predictions based on 'alpha'
    PASTEL_combined_pred_train = (alpha[ats_label] * local_pred_train) + ((1 - alpha[ats_label]) * local_pred_train) 
    PASTEL_combined_pred_test = (alpha[ats_label] * global_pred_test) + ((1 - alpha[ats_label]) * local_pred_test)

    # Perform log transformation on training and testing set if specified
    if log_transform[ats_label]:
        PASTEL_combined_pred_train_i = np.expm1(PASTEL_combined_pred_train)
        PASTEL_combined_pred_test_i = np.expm1(PASTEL_combined_pred_test)
    else:
        PASTEL_combined_pred_train_i = PASTEL_combined_pred_train
        PASTEL_combined_pred_test_i = PASTEL_combined_pred_test

    # Calcuate the combined model residuals
    combined_residuals_train = y_train_i - PASTEL_combined_pred_train_i
    combined_residuals_test = y_test_i - PASTEL_combined_pred_test_i

    # Save the combined model predictions and residuals to use later
    np.save(os.path.join(combined_save_path, f"{ats_label}_PASTEL_combined_pred_train.npy"), PASTEL_combined_pred_train_i)
    np.save(os.path.join(combined_save_path, f"{ats_label}_PASTEL_combined_pred_test.npy"), PASTEL_combined_pred_test_i)
    np.save(os.path.join(combined_save_path, f"{ats_label}_PASTEL_combined_residuals_train.npy"), combined_residuals_train)
    np.save(os.path.join(combined_save_path, f"{ats_label}_PASTEL_combined_residuals_test.npy"), combined_residuals_test)

    # Visulize predictions on training set
    title = f'Training Results: {ats_label} [PASTEL - combined]'
    scatter_plot(y_train_i, PASTEL_combined_pred_train_i, title, ats_label, unit_label, False, os.path.join(combined_save_path, f"PASTEL_{ats_label}_train.png"))
    # Visulize predictions on testing set
    title = f'Testing Results: {ats_label} [PASTEL - combined]'
    scatter_plot(y_test_i, PASTEL_combined_pred_test_i, title, ats_label, unit_label, False, os.path.join(combined_save_path, f"PASTEL_{ats_label}_test.png"))

    print(f"Finished Run for ATS: {ats_label}")

print("Annnnnnnd that's a wrap....")

In [ ]:
# Quick plot to see the MWFL curve for checking values
# w = 1
# n_values = np.arange(1, 1001, 10)  # Range of sample sizes, from 1 to 1000, step of 10

# # Compute mwfl for each value of n
# mwfl_values = w / n_values

# # Value of mwfl at n=10 
# n_annotation = 100
# mwfl_annotation = w / n_annotation

# # Plotting
# plt.figure(figsize=(10, 6))
# plt.plot(n_values, mwfl_values, label='mwfl = ε / n\n ε = 1', color='b')

# # Add annotation at n=10
# plt.annotate(
#     f'mwfl at n={n_annotation} = {mwfl_annotation:.3f}',
#     xy=(n_annotation, mwfl_annotation),
#     xytext=(n_annotation + 100, mwfl_annotation + 0.05),  # Offset for clarity
#     arrowprops=dict(arrowstyle='->', color='red'),
#     fontsize=10,
#     color='red'
# )

# # Labels and Title
# plt.xlabel('Sample Size (n)')
# plt.ylabel('Min Weight Fraction Leaf (mwfl)')
# plt.title('\"mwfl\" as Sample Size \"n\" Increases')
# plt.legend()
# plt.grid(True)
# plt.show()